In [1]:
# =========================================================================
# VALIDATION PIPELINE (translation already done) -- Hindi -> English
# Merges your existing translations with the gold English references from
# your aligned bulletin file, then scores every advisory section.
#
# WHY THIS VERSION IS DIFFERENT:
#   - You already have mt_english for all 16 sections (translated_by_section.xlsx).
#     This script does NOT re-translate anything -- it skips straight to
#     merging gold references + scoring, so it's much faster than the
#     translate+validate pipeline.
#   - Your gold references live in a WIDE-format file (aligned_advisory_text.xlsx,
#     one row per bulletin with "<section> [EN]" / "<section> [HI]" column pairs).
#     This script reshapes that into gold text per section and joins it onto
#     your long-format translated data using state + district + month + date
#     as the match key.
#   - All metrics (BERTScore, COMET, LaBSE cosine, chrF++, BLEU, etc.) compare
#     mt_english against english_gold -- i.e. English vs English, same
#     language on both sides, every time. That's what makes these scores
#     valid in the first place.
#   - LANGUAGE CONFIG is factored out (see LANGUAGE section below) so if you
#     later run this on a different source language (e.g. Punjabi), you only
#     need to change SOURCE_LANG_NAME / SOURCE_NLLB_CODE -- the merge and
#     scoring logic underneath doesn't change, since it only ever operates
#     on the English (mt_english vs english_gold) side.
#
# Run in Google Colab. GPU (T4) makes BERTScore/COMET/LaBSE faster but this
# will also run on CPU (just slower) -- auto-detected below.
#
# Upload BOTH files to the Colab session first:
#   - translated_by_section.xlsx   (your existing translations, all sheets)
#   - aligned_advisory_text.xlsx   (the aligned file with EN/HI column pairs)
# =========================================================================

!pip install -q sacrebleu rouge-score nltk bert-score sentence-transformers unbabel-comet openpyxl

import os
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import re
import gc
import time
import pandas as pd
import numpy as np
import torch
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
if device == "cpu":
    print("No GPU detected -- will still run, just slower for BERTScore/COMET/LaBSE.")

# =========================================================================
# OPTIONAL: mount Google Drive so outputs survive a runtime reset
# =========================================================================
MOUNT_DRIVE = True
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/agromet_pipeline_outputs"

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    OUT_DIR = DRIVE_OUTPUT_DIR
else:
    OUT_DIR = "."
print(f"Outputs will be saved to: {OUT_DIR}")


# =========================================================================
# LANGUAGE CONFIG -- change these two lines to extend this pipeline to a
# different source language later (e.g. Punjabi). Nothing else in this
# script needs to change, since scoring always happens mt_english vs
# english_gold (English vs English).
# =========================================================================
SOURCE_LANG_NAME = "Hindi"
TARGET_LANG_CODE = "en"          # used by BERTScore -- keep "en" since target is always English here


# =========================================================================
# CONFIG
# =========================================================================
CONFIG = {
    "TRANSLATED_FILE": "translated_by_section.xlsx",   # your existing translations (long format, per-sheet)
    "ALIGNED_FILE": "aligned_advisory_text.xlsx",       # gold references (wide format, EN/HI pairs)
    "ALIGNED_SHEET": "advisory_paired",

    "RESULTS_OUTPUT_FILE": os.path.join(OUT_DIR, "validation_results_all_sections.xlsx"),
    "REPORT_OUTPUT_FILE": os.path.join(OUT_DIR, "metric_accuracy_by_section.xlsx"),
    "MERGED_FILE": os.path.join(OUT_DIR, "merged_with_gold_all_sections.xlsx"),

    "SOURCE_COL": "hindi_text",
    "MT_COL": "mt_english",
    "REF_COL": "english_gold",

    "JOIN_KEYS": ["state", "district", "month", "date"],

    # translated-sheet-name -> aligned-file-column-prefix
    "SECTION_MAPPING": {
        "forecast_summary": "forecast_summary",
        "bulletin_subtitle": "bulletin_subtitle",
        "weather_warnings": "weather_warnings",
        "weather_warning_impacts": "weather_warning_impacts",
        "weather_impacts_general": "weather_impacts_general",
        "general_advisory": "general_advisory",
        "sms_advisory": "sms_advisory",
        "impact_based_advisories": "impact_based_advisories_general",
        "crop_advisory": "crop_advisory",
        "horticulture_advisory": "horticulture_advisory",
        "livestock_advisory": "livestock_advisory",
        "fisheries_advisory": "fisheries_advisory",
        "poultry_advisory": "poultry_advisory",
        "masthead": "__masthead__",
        "preamble": "__preamble__",
        "unmatched_other": "unmatched_other",
    },

    "BERTSCORE_MODEL": "microsoft/deberta-large-mnli",
    "BERTSCORE_BATCH_SIZE": 8 if device == "cuda" else 4,

    "RUN_COMET": True,   # set False to skip COMET (it's the slowest model, esp. on CPU)

    # --- flagging thresholds (calibrated against forecast_summary run) ---
    "THRESH_BERTSCORE_F1": 0.80,
    "THRESH_COSINE": 0.75,
    "THRESH_COMET": 0.75,
    "THRESH_CHRF": 40.0,
    "THRESH_NUMERIC_CONSISTENCY": 0.5,
    "THRESH_ENTITY_PRESERVATION": 0.4,
}

if device == "cpu" and CONFIG["RUN_COMET"]:
    print("NOTE: COMET on CPU is slow (can take a long time across ~5000 rows). "
          "Set CONFIG['RUN_COMET'] = False above if you want to skip it and rely on "
          "BERTScore + LaBSE cosine instead.")


# =========================================================================
# 1. MERGE: attach english_gold from the aligned wide-format file onto
#    every sheet of the already-translated long-format file
# =========================================================================
def merge_gold_references(cfg):
    aligned = pd.read_excel(cfg["ALIGNED_FILE"], sheet_name=cfg["ALIGNED_SHEET"])
    xl = pd.ExcelFile(cfg["TRANSLATED_FILE"])

    frames = []
    print(f"Sections found in translated file: {xl.sheet_names}\n")
    for sheet in xl.sheet_names:
        prefix = cfg["SECTION_MAPPING"].get(sheet)
        if prefix is None:
            print(f"  Skipping '{sheet}': no mapping to an aligned-file column.")
            continue
        en_col = f"{prefix} [EN]"
        if en_col not in aligned.columns:
            print(f"  Skipping '{sheet}': column '{en_col}' not found in aligned file.")
            continue

        df = xl.parse(sheet)
        gold_sub = aligned[cfg["JOIN_KEYS"] + [en_col]].rename(columns={en_col: cfg["REF_COL"]})
        # de-dupe in case the aligned file has repeated keys
        gold_sub = gold_sub.drop_duplicates(subset=cfg["JOIN_KEYS"], keep="first")

        merged = df.merge(gold_sub, on=cfg["JOIN_KEYS"], how="left")
        merged[cfg["REF_COL"]] = merged[cfg["REF_COL"]].astype(str)

        # has_pair: real gold text present (not NaN/'nan'/empty/whitespace-only)
        cleaned = merged[cfg["REF_COL"]].str.strip()
        merged["has_pair"] = ~cleaned.isin(["", "nan", "None", "NaN"])

        merged["field_name"] = sheet
        merged["sheet"] = sheet

        n_paired = merged["has_pair"].sum()
        print(f"  {sheet:32s} rows={len(merged):5d}  paired={n_paired:5d} ({n_paired/len(merged)*100:.1f}%)")
        frames.append(merged)

    full = pd.concat(frames, ignore_index=True)
    full[cfg["SOURCE_COL"]] = full[cfg["SOURCE_COL"]].astype(str)
    full[cfg["MT_COL"]] = full[cfg["MT_COL"]].astype(str)

    with pd.ExcelWriter(cfg["MERGED_FILE"], engine="openpyxl") as writer:
        for sheet in full["field_name"].unique():
            full[full["field_name"] == sheet].to_excel(writer, sheet_name=sheet[:31], index=False)
    print(f"\nSaved {cfg['MERGED_FILE']}")

    total_paired = full["has_pair"].sum()
    print(f"\nTotal rows: {len(full)}  |  Total scoreable (paired) rows: {total_paired} "
          f"({total_paired/len(full)*100:.1f}%)")
    return full


# =========================================================================
# 2. N-GRAM / EDIT-DISTANCE METRICS (diagnostic only)
# =========================================================================
def compute_ngram_metrics(df, cfg):
    import sacrebleu
    from rouge_score import rouge_scorer
    import nltk
    nltk.download("wordnet", quiet=True)
    nltk.download("omw-1.4", quiet=True)
    nltk.download("punkt", quiet=True)
    nltk.download("punkt_tab", quiet=True)
    from nltk.translate.meteor_score import meteor_score

    rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    scoreable = df[df["has_pair"] == True]
    print(f"  Scoring {len(scoreable)} rows (BLEU/chrF++/TER/ROUGE-L/METEOR)...")

    bleu, chrf, ter, rougeL, meteor = {}, {}, {}, {}, {}
    start = time.time()
    for n, (idx, row) in enumerate(scoreable.iterrows(), 1):
        mt, ref = row[cfg["MT_COL"]], row[cfg["REF_COL"]]
        bleu[idx] = sacrebleu.sentence_bleu(mt, [ref]).score
        chrf[idx] = sacrebleu.sentence_chrf(mt, [ref], word_order=2).score
        ter[idx] = sacrebleu.sentence_ter(mt, [ref]).score
        rougeL[idx] = rouge.score(ref, mt)["rougeL"].fmeasure
        try:
            meteor[idx] = meteor_score([ref.split()], mt.split())
        except Exception:
            meteor[idx] = np.nan
        if n % 200 == 0 or n == len(scoreable):
            rate = n / (time.time() - start) if time.time() > start else 0
            print(f"    {n}/{len(scoreable)}  ({rate:.1f} rows/sec)")

    df["bleu"] = df.index.map(bleu)
    df["chrf++"] = df.index.map(chrf)
    df["ter"] = df.index.map(ter)
    df["rouge_l"] = df.index.map(rougeL)
    df["meteor"] = df.index.map(meteor)
    return df


# =========================================================================
# 3. BERTSCORE
# =========================================================================
def compute_bertscore(df, cfg):
    from bert_score import score as bertscore
    scoreable = df[df["has_pair"] == True]
    P, R, F1 = bertscore(
        scoreable[cfg["MT_COL"]].tolist(), scoreable[cfg["REF_COL"]].tolist(),
        lang=TARGET_LANG_CODE, model_type=cfg["BERTSCORE_MODEL"],
        batch_size=cfg["BERTSCORE_BATCH_SIZE"], device=device, verbose=True
    )
    df.loc[scoreable.index, "bertscore_precision"] = P.numpy()
    df.loc[scoreable.index, "bertscore_recall"] = R.numpy()
    df.loc[scoreable.index, "bertscore_f1"] = F1.numpy()
    free_gpu()
    return df


# =========================================================================
# 4. LaBSE COSINE SIMILARITY
# =========================================================================
def compute_cosine_labse(df, cfg):
    from sentence_transformers import SentenceTransformer, util
    labse = SentenceTransformer("sentence-transformers/LaBSE", device=device)
    scoreable = df[df["has_pair"] == True]

    emb_mt = labse.encode(scoreable[cfg["MT_COL"]].tolist(), batch_size=32, show_progress_bar=True, convert_to_tensor=True)
    emb_ref = labse.encode(scoreable[cfg["REF_COL"]].tolist(), batch_size=32, show_progress_bar=True, convert_to_tensor=True)
    emb_src = labse.encode(scoreable[cfg["SOURCE_COL"]].tolist(), batch_size=32, show_progress_bar=True, convert_to_tensor=True)

    df.loc[scoreable.index, "cosine_mt_vs_ref"] = util.pairwise_cos_sim(emb_mt, emb_ref).cpu().numpy()
    df.loc[scoreable.index, "cosine_src_vs_ref_direct"] = util.pairwise_cos_sim(emb_src, emb_ref).cpu().numpy()
    del labse
    free_gpu()
    return df


# =========================================================================
# 5. COMET
# =========================================================================
def compute_comet(df, cfg):
    if not cfg["RUN_COMET"]:
        print("  Skipping COMET (CONFIG['RUN_COMET'] = False).")
        return df
    from comet import download_model, load_from_checkpoint
    scoreable = df[df["has_pair"] == True]
    print("Loading COMET (reference-based)...")
    path = download_model("Unbabel/wmt22-comet-da")
    model = load_from_checkpoint(path)
    data = [{"src": s, "mt": m, "ref": r} for s, m, r in
            zip(scoreable[cfg["SOURCE_COL"]], scoreable[cfg["MT_COL"]], scoreable[cfg["REF_COL"]])]
    out = model.predict(data, batch_size=8, gpus=1 if device == "cuda" else 0)
    df.loc[scoreable.index, "comet"] = out.scores
    del model
    free_gpu()
    return df


# =========================================================================
# 6. ENTITY / NUMERIC / STRUCTURAL CHECKS
# =========================================================================
NUM_RE = re.compile(r"\d+(?:\.\d+)?")
DEVANAGARI_RE = re.compile(r"[\u0900-\u097F]")
CAP_ENTITY_RE = re.compile(r"\b[A-Z][A-Za-z]{2,}\b")

def extract_numbers(text):
    return set(NUM_RE.findall(text))

def numeric_consistency(src, mt):
    src_nums = extract_numbers(src)
    if not src_nums:
        return 1.0
    return len(src_nums & extract_numbers(mt)) / len(src_nums)

def extract_capitalized_entities(text):
    return set(w.upper() for w in CAP_ENTITY_RE.findall(text))

def entity_preservation(ref, mt):
    ref_ents = extract_capitalized_entities(ref)
    if not ref_ents:
        return 1.0
    return len(ref_ents & extract_capitalized_entities(mt)) / len(ref_ents)

def has_untranslated_source_script(mt_text):
    return bool(DEVANAGARI_RE.search(mt_text))

def length_ratio(src, mt):
    src_len = max(len(src.split()), 1)
    return len(mt.split()) / src_len

def compute_structural_checks(df, cfg):
    df["numeric_consistency"] = df.apply(
        lambda r: numeric_consistency(r[cfg["SOURCE_COL"]], r[cfg["MT_COL"]]), axis=1)
    scoreable = df[df["has_pair"] == True]
    df.loc[scoreable.index, "entity_preservation"] = scoreable.apply(
        lambda r: entity_preservation(r[cfg["REF_COL"]], r[cfg["MT_COL"]]), axis=1)
    df["has_untranslated_text"] = df[cfg["MT_COL"]].apply(has_untranslated_source_script)
    df["length_ratio"] = df.apply(
        lambda r: length_ratio(r[cfg["SOURCE_COL"]], r[cfg["MT_COL"]]), axis=1)
    return df


# =========================================================================
# 7. FLAGS (per-row, no aggregate needs_review column)
# =========================================================================
def compute_flags(df, cfg):
    if "bertscore_f1" in df:
        df["flag_low_bertscore"] = df["bertscore_f1"] < cfg["THRESH_BERTSCORE_F1"]
    if "cosine_mt_vs_ref" in df:
        df["flag_low_cosine"] = df["cosine_mt_vs_ref"] < cfg["THRESH_COSINE"]
    if "comet" in df:
        df["flag_low_comet"] = df["comet"] < cfg["THRESH_COMET"]
    if "chrf++" in df:
        df["flag_low_chrf"] = df["chrf++"] < cfg["THRESH_CHRF"]
    if "has_untranslated_text" in df:
        df["flag_untranslated_text"] = df["has_untranslated_text"]
    if "numeric_consistency" in df:
        df["flag_low_numeric_consistency"] = df["numeric_consistency"] < cfg["THRESH_NUMERIC_CONSISTENCY"]
    if "entity_preservation" in df:
        df["flag_low_entity_preservation"] = df["entity_preservation"] < cfg["THRESH_ENTITY_PRESERVATION"]
    return df


# =========================================================================
# 8. FINAL REPORT: metric_accuracy_by_section.xlsx
# =========================================================================
def build_accuracy_report(df, cfg):
    scored = df[df["has_pair"] == True].copy()
    if len(scored) == 0:
        print("No scoreable rows -- skipping report.")
        return

    flagged_metrics = [
        ("chrf++", cfg["THRESH_CHRF"]),
        ("bertscore_f1", cfg["THRESH_BERTSCORE_F1"]),
        ("cosine_mt_vs_ref", cfg["THRESH_COSINE"]),
        ("numeric_consistency", cfg["THRESH_NUMERIC_CONSISTENCY"]),
        ("entity_preservation", cfg["THRESH_ENTITY_PRESERVATION"]),
    ]
    if cfg["RUN_COMET"]:
        flagged_metrics.insert(3, ("comet", cfg["THRESH_COMET"]))

    diagnostic_metrics = ["bleu", "ter", "rouge_l", "meteor", "cosine_src_vs_ref_direct", "length_ratio"]
    sections = sorted(scored["field_name"].unique())

    acc_rows = []
    for sec in sections:
        sub = scored[scored["field_name"] == sec]
        row = {"Section (Advisory)": sec, "n": len(sub)}
        for metric, thresh in flagged_metrics:
            if metric in sub.columns:
                row[metric] = round((sub[metric] >= thresh).mean() * 100, 1)
        acc_rows.append(row)
    acc_df = pd.DataFrame(acc_rows).set_index("Section (Advisory)")

    all_metric_cols = [m for m, _ in flagged_metrics] + diagnostic_metrics
    med_rows = []
    for sec in sections:
        sub = scored[scored["field_name"] == sec]
        row = {"Section (Advisory)": sec, "n": len(sub)}
        for metric in all_metric_cols:
            if metric in sub.columns:
                row[metric] = round(sub[metric].median(), 3)
        med_rows.append(row)
    med_df = pd.DataFrame(med_rows).set_index("Section (Advisory)")

    wb = Workbook()
    arial = "Arial"
    title_font = Font(name=arial, size=13, bold=True)
    header_font = Font(name=arial, size=10, bold=True)
    body_font = Font(name=arial, size=10)
    wrap = Alignment(wrap_text=True, vertical="top", horizontal="left")
    center = Alignment(wrap_text=True, vertical="top", horizontal="center")
    thin = Side(style="thin", color="000000")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)

    def write_table(ws, title, table_df, subtitle=""):
        ws.merge_cells(f"A1:{get_column_letter(len(table_df.columns)+1)}1")
        ws["A1"] = title
        ws["A1"].font = title_font
        ws.row_dimensions[1].height = 22
        r0 = 2
        if subtitle:
            ws.merge_cells(f"A2:{get_column_letter(len(table_df.columns)+1)}2")
            ws["A2"] = subtitle
            ws["A2"].font = Font(name=arial, size=9, italic=True)
            r0 = 3
        headers = ["Section (Advisory)"] + list(table_df.columns)
        for c, h in enumerate(headers, start=1):
            cell = ws.cell(row=r0, column=c, value=h)
            cell.font = header_font
            cell.border = border
            cell.alignment = center
        r = r0 + 1
        for idx, row in table_df.iterrows():
            ws.cell(row=r, column=1, value=idx).font = body_font
            ws.cell(row=r, column=1).border = border
            ws.cell(row=r, column=1).alignment = wrap
            for c, val in enumerate(row, start=2):
                cell = ws.cell(row=r, column=c, value=val)
                cell.font = body_font
                cell.border = border
                cell.alignment = center
            r += 1
        widths = [26] + [16] * len(table_df.columns)
        for i, w in enumerate(widths, start=1):
            ws.column_dimensions[get_column_letter(i)].width = w
        ws.freeze_panes = f"A{r0+1}"

    ws1 = wb.active
    ws1.title = "Accuracy % by Section"
    write_table(ws1, f"Accuracy % by Advisory Section ({SOURCE_LANG_NAME} -> English)",
                acc_df, "Each cell = % of that section's rows meeting the metric's accuracy threshold.")

    ws2 = wb.create_sheet("Median Scores by Section")
    write_table(ws2, "Median Raw Score by Advisory Section (per metric)",
                med_df, "Diagnostic metrics shown as median only -- not used for pass/fail.")

    ws3 = wb.create_sheet("Metric Definitions")
    ws3["A1"] = "Metric Definitions and Thresholds"
    ws3["A1"].font = title_font
    defs = [
        ("bleu", "0-100", "Diagnostic only", "Word n-gram overlap. Naturally low for paraphrastic MT."),
        ("chrf++", "0-100", f">= {cfg['THRESH_CHRF']}", "Character n-gram overlap; more forgiving than BLEU."),
        ("ter", "0-100+ (lower=better)", "Diagnostic only", "Edit distance to fix MT into reference."),
        ("rouge_l", "0-1", "Diagnostic only", "Longest common word-sequence overlap."),
        ("meteor", "0-1", "Diagnostic only", "Overlap with stemming/synonym matching."),
        ("bertscore_f1", "0-1", f">= {cfg['THRESH_BERTSCORE_F1']}", "Embedding-based semantic overlap."),
        ("cosine_mt_vs_ref", "0-1 (theoretical -1 to 1)", f">= {cfg['THRESH_COSINE']}", "LaBSE meaning-similarity, MT vs gold reference."),
        ("cosine_src_vs_ref_direct", "0-1 (theoretical -1 to 1)", "Diagnostic only (checks reference, not model)", f"LaBSE similarity, {SOURCE_LANG_NAME} source vs gold reference directly."),
        ("comet", "Roughly 0-1", f">= {cfg['THRESH_COMET']}" if cfg["RUN_COMET"] else "Skipped this run", "Trained quality-estimation model; closest to human judgment."),
        ("numeric_consistency", "0-1", f">= {cfg['THRESH_NUMERIC_CONSISTENCY']}", "Fraction of source numbers preserved in MT."),
        ("entity_preservation", "0-1", f">= {cfg['THRESH_ENTITY_PRESERVATION']}", "Fraction of reference's named entities found in MT."),
        ("length_ratio", "Roughly 0-2 (1.0=same length)", "Diagnostic only", "MT word count / source word count."),
    ]
    headers = ["Metric", "Range", "Accuracy Threshold", "What It Measures"]
    for c, h in enumerate(headers, start=1):
        cell = ws3.cell(row=3, column=c, value=h)
        cell.font = header_font
        cell.border = border
        cell.alignment = center
    r = 4
    for metric, scale, thresh, desc in defs:
        vals = [metric, scale, thresh, desc]
        for c, v in enumerate(vals, start=1):
            cell = ws3.cell(row=r, column=c, value=v)
            cell.font = body_font
            cell.border = border
            cell.alignment = wrap
        ws3.row_dimensions[r].height = 32
        r += 1
    widths = [22, 26, 30, 55]
    for i, w in enumerate(widths, start=1):
        ws3.column_dimensions[get_column_letter(i)].width = w
    ws3.freeze_panes = "A4"

    wb.save(cfg["REPORT_OUTPUT_FILE"])
    print(f"\nSaved {cfg['REPORT_OUTPUT_FILE']}")
    print("\n--- Accuracy % by Section ---")
    print(acc_df)


# =========================================================================
# MAIN
# =========================================================================
def run_validation_pipeline(cfg):
    overall_start = time.time()

    print("=== STEP 1: Merging gold references into translated data ===")
    df = merge_gold_references(cfg)

    print("\n=== STEP 2: N-gram / edit-distance metrics (diagnostic) ===")
    df = compute_ngram_metrics(df, cfg)

    print("\n=== STEP 3: BERTScore ===")
    df = compute_bertscore(df, cfg)

    print("\n=== STEP 4: LaBSE cosine similarity ===")
    df = compute_cosine_labse(df, cfg)

    print("\n=== STEP 5: COMET ===")
    df = compute_comet(df, cfg)

    print("\n=== STEP 6: Entity / numeric / structural checks ===")
    df = compute_structural_checks(df, cfg)

    print("\n=== STEP 7: Flags ===")
    df = compute_flags(df, cfg)

    df.to_excel(cfg["RESULTS_OUTPUT_FILE"], index=False)
    print(f"Saved {cfg['RESULTS_OUTPUT_FILE']}")

    print("\n=== STEP 8: Building per-section accuracy report ===")
    build_accuracy_report(df, cfg)

    print(f"\nAll done in {(time.time()-overall_start)/60:.1f} min total.")
    return df


# ---- RUN ----
final_df = run_validation_pipeline(CONFIG)

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.0/91.0 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.4/101.4 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 97.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.7/529.7 kB 31.9 MB/s eta 0:00:00

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [1]:
# =========================================================================
# CELL 1 -- RUN THIS FIRST, THEN RESTART THE RUNTIME
#
# WHY: unbabel-comet pulls in numpy<2 as a dependency. Colab's base image
# ships with numpy 2.x, and pandas/scipy/etc. are compiled against that.
# If pip silently downgrades numpy to satisfy comet, every other library
# that expects numpy 2's memory layout breaks with a "dtype size changed"
# error the moment you import pandas. Installing comet with --no-deps
# avoids the downgrade entirely (comet's actual code runs fine on numpy 2
# despite the overly strict version pin in its metadata).
#
# After this cell finishes:
#   Runtime menu -> Restart session (NOT "Restart and run all")
#   Then run CELL 2 in a fresh cell below. Do not re-run this cell after
#   restarting -- the packages are already installed and will persist.
# =========================================================================

!pip install -q --no-deps unbabel-comet
!pip install -q "pytorch-lightning>=2.0" "jsonargparse[signatures]>=4.27.5" "omegaconf>=2.3.0"
!pip install -q transformers sentencepiece accelerate sacrebleu rouge-score nltk bert-score sentence-transformers openpyxl

# Confirm numpy was NOT downgraded -- should print 2.x
import numpy
print("\nnumpy version after install:", numpy.__version__)
if numpy.__version__.startswith("1."):
    print("WARNING: numpy was still downgraded to 1.x. Run the cell below to force it back to 2.x, "
          "then restart the runtime before proceeding to Cell 2.")
    print('!pip install -q --upgrade --no-deps "numpy>=2"')
else:
    print("Good -- numpy is still 2.x. Now restart the runtime (Runtime > Restart session), "
          "then run Cell 2 below.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.9/796.9 kB 30.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unbabel-comet 2.2.7 requires jsonargparse==3.13.1, but you have jsonargparse 4.49.0 which is incompatible.

numpy version after install: 1.26.4
!pip install -q --upgrade --no-deps "numpy>=2"


In [2]:
# =========================================================================
# CELL 2 -- run this AFTER Cell 1 finished AND you restarted the runtime.
# Do not run Cell 1's pip installs again in this session -- they already
# persisted through the restart.
# =========================================================================
#
# VALIDATION PIPELINE (translation already done) -- Hindi -> English
# Merges your existing translations with the gold English references from
# your aligned bulletin file, then scores every advisory section.
#
# WHY THIS VERSION IS DIFFERENT:
#   - You already have mt_english for all 16 sections (translated_by_section.xlsx).
#     This script does NOT re-translate anything -- it skips straight to
#     merging gold references + scoring, so it's much faster than the
#     translate+validate pipeline.
#   - Your gold references live in a WIDE-format file (aligned_advisory_text.xlsx,
#     one row per bulletin with "<section> [EN]" / "<section> [HI]" column pairs).
#     This script reshapes that into gold text per section and joins it onto
#     your long-format translated data using state + district + month + date
#     as the match key.
#   - All metrics (BERTScore, COMET, LaBSE cosine, chrF++, BLEU, etc.) compare
#     mt_english against english_gold -- i.e. English vs English, same
#     language on both sides, every time. That's what makes these scores
#     valid in the first place.
#   - LANGUAGE CONFIG is factored out (see LANGUAGE section below) so if you
#     later run this on a different source language (e.g. Punjabi), you only
#     need to change SOURCE_LANG_NAME / SOURCE_NLLB_CODE -- the merge and
#     scoring logic underneath doesn't change, since it only ever operates
#     on the English (mt_english vs english_gold) side.
#
# Run in Google Colab. GPU (T4) makes BERTScore/COMET/LaBSE faster but this
# will also run on CPU (just slower) -- auto-detected below.
#
# Upload BOTH files to the Colab session first:
#   - translated_by_section.xlsx   (your existing translations, all sheets)
#   - aligned_advisory_text.xlsx   (the aligned file with EN/HI column pairs)
# =========================================================================


import os
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import re
import gc
import time
import pandas as pd
import numpy as np
import torch
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
if device == "cpu":
    print("No GPU detected -- will still run, just slower for BERTScore/COMET/LaBSE.")

# =========================================================================
# OPTIONAL: mount Google Drive so outputs survive a runtime reset
# =========================================================================
MOUNT_DRIVE = True
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/agromet_pipeline_outputs"

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    OUT_DIR = DRIVE_OUTPUT_DIR
else:
    OUT_DIR = "."
print(f"Outputs will be saved to: {OUT_DIR}")


# =========================================================================
# LANGUAGE CONFIG -- change these two lines to extend this pipeline to a
# different source language later (e.g. Punjabi). Nothing else in this
# script needs to change, since scoring always happens mt_english vs
# english_gold (English vs English).
# =========================================================================
SOURCE_LANG_NAME = "Hindi"
TARGET_LANG_CODE = "en"          # used by BERTScore -- keep "en" since target is always English here


# =========================================================================
# CONFIG
# =========================================================================
CONFIG = {
    "TRANSLATED_FILE": "translated_by_section.xlsx",   # your existing translations (long format, per-sheet)
    "ALIGNED_FILE": "aligned_advisory_text.xlsx",       # gold references (wide format, EN/HI pairs)
    "ALIGNED_SHEET": "advisory_paired",

    "RESULTS_OUTPUT_FILE": os.path.join(OUT_DIR, "validation_results_all_sections.xlsx"),
    "REPORT_OUTPUT_FILE": os.path.join(OUT_DIR, "metric_accuracy_by_section.xlsx"),
    "MERGED_FILE": os.path.join(OUT_DIR, "merged_with_gold_all_sections.xlsx"),

    "SOURCE_COL": "hindi_text",
    "MT_COL": "mt_english",
    "REF_COL": "english_gold",

    "JOIN_KEYS": ["state", "district", "month", "date"],

    # translated-sheet-name -> aligned-file-column-prefix
    "SECTION_MAPPING": {
        "forecast_summary": "forecast_summary",
        "bulletin_subtitle": "bulletin_subtitle",
        "weather_warnings": "weather_warnings",
        "weather_warning_impacts": "weather_warning_impacts",
        "weather_impacts_general": "weather_impacts_general",
        "general_advisory": "general_advisory",
        "sms_advisory": "sms_advisory",
        "impact_based_advisories": "impact_based_advisories_general",
        "crop_advisory": "crop_advisory",
        "horticulture_advisory": "horticulture_advisory",
        "livestock_advisory": "livestock_advisory",
        "fisheries_advisory": "fisheries_advisory",
        "poultry_advisory": "poultry_advisory",
        "masthead": "__masthead__",
        "preamble": "__preamble__",
        "unmatched_other": "unmatched_other",
    },

    "BERTSCORE_MODEL": "microsoft/deberta-large-mnli",
    "BERTSCORE_BATCH_SIZE": 8 if device == "cuda" else 4,

    "RUN_COMET": True,   # set False to skip COMET (it's the slowest model, esp. on CPU)

    # --- flagging thresholds (calibrated against forecast_summary run) ---
    "THRESH_BERTSCORE_F1": 0.80,
    "THRESH_COSINE": 0.75,
    "THRESH_COMET": 0.75,
    "THRESH_CHRF": 40.0,
    "THRESH_NUMERIC_CONSISTENCY": 0.5,
    "THRESH_ENTITY_PRESERVATION": 0.4,
}

if device == "cpu" and CONFIG["RUN_COMET"]:
    print("NOTE: COMET on CPU is slow (can take a long time across ~5000 rows). "
          "Set CONFIG['RUN_COMET'] = False above if you want to skip it and rely on "
          "BERTScore + LaBSE cosine instead.")


# =========================================================================
# 1. MERGE: attach english_gold from the aligned wide-format file onto
#    every sheet of the already-translated long-format file
# =========================================================================
def merge_gold_references(cfg):
    aligned = pd.read_excel(cfg["ALIGNED_FILE"], sheet_name=cfg["ALIGNED_SHEET"])
    xl = pd.ExcelFile(cfg["TRANSLATED_FILE"])

    frames = []
    print(f"Sections found in translated file: {xl.sheet_names}\n")
    for sheet in xl.sheet_names:
        prefix = cfg["SECTION_MAPPING"].get(sheet)
        if prefix is None:
            print(f"  Skipping '{sheet}': no mapping to an aligned-file column.")
            continue
        en_col = f"{prefix} [EN]"
        if en_col not in aligned.columns:
            print(f"  Skipping '{sheet}': column '{en_col}' not found in aligned file.")
            continue

        df = xl.parse(sheet)
        gold_sub = aligned[cfg["JOIN_KEYS"] + [en_col]].rename(columns={en_col: cfg["REF_COL"]})
        # de-dupe in case the aligned file has repeated keys
        gold_sub = gold_sub.drop_duplicates(subset=cfg["JOIN_KEYS"], keep="first")

        merged = df.merge(gold_sub, on=cfg["JOIN_KEYS"], how="left")
        merged[cfg["REF_COL"]] = merged[cfg["REF_COL"]].astype(str)

        # has_pair: real gold text present (not NaN/'nan'/empty/whitespace-only)
        cleaned = merged[cfg["REF_COL"]].str.strip()
        merged["has_pair"] = ~cleaned.isin(["", "nan", "None", "NaN"])

        merged["field_name"] = sheet
        merged["sheet"] = sheet

        n_paired = merged["has_pair"].sum()
        print(f"  {sheet:32s} rows={len(merged):5d}  paired={n_paired:5d} ({n_paired/len(merged)*100:.1f}%)")
        frames.append(merged)

    full = pd.concat(frames, ignore_index=True)
    full[cfg["SOURCE_COL"]] = full[cfg["SOURCE_COL"]].astype(str)
    full[cfg["MT_COL"]] = full[cfg["MT_COL"]].astype(str)

    with pd.ExcelWriter(cfg["MERGED_FILE"], engine="openpyxl") as writer:
        for sheet in full["field_name"].unique():
            full[full["field_name"] == sheet].to_excel(writer, sheet_name=sheet[:31], index=False)
    print(f"\nSaved {cfg['MERGED_FILE']}")

    total_paired = full["has_pair"].sum()
    print(f"\nTotal rows: {len(full)}  |  Total scoreable (paired) rows: {total_paired} "
          f"({total_paired/len(full)*100:.1f}%)")
    return full


# =========================================================================
# 2. N-GRAM / EDIT-DISTANCE METRICS (diagnostic only)
# =========================================================================
def compute_ngram_metrics(df, cfg):
    import sacrebleu
    from rouge_score import rouge_scorer
    import nltk
    nltk.download("wordnet", quiet=True)
    nltk.download("omw-1.4", quiet=True)
    nltk.download("punkt", quiet=True)
    nltk.download("punkt_tab", quiet=True)
    from nltk.translate.meteor_score import meteor_score

    rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    scoreable = df[df["has_pair"] == True]
    print(f"  Scoring {len(scoreable)} rows (BLEU/chrF++/TER/ROUGE-L/METEOR)...")

    bleu, chrf, ter, rougeL, meteor = {}, {}, {}, {}, {}
    start = time.time()
    for n, (idx, row) in enumerate(scoreable.iterrows(), 1):
        mt, ref = row[cfg["MT_COL"]], row[cfg["REF_COL"]]
        bleu[idx] = sacrebleu.sentence_bleu(mt, [ref]).score
        chrf[idx] = sacrebleu.sentence_chrf(mt, [ref], word_order=2).score
        ter[idx] = sacrebleu.sentence_ter(mt, [ref]).score
        rougeL[idx] = rouge.score(ref, mt)["rougeL"].fmeasure
        try:
            meteor[idx] = meteor_score([ref.split()], mt.split())
        except Exception:
            meteor[idx] = np.nan
        if n % 200 == 0 or n == len(scoreable):
            rate = n / (time.time() - start) if time.time() > start else 0
            print(f"    {n}/{len(scoreable)}  ({rate:.1f} rows/sec)")

    df["bleu"] = df.index.map(bleu)
    df["chrf++"] = df.index.map(chrf)
    df["ter"] = df.index.map(ter)
    df["rouge_l"] = df.index.map(rougeL)
    df["meteor"] = df.index.map(meteor)
    return df


# =========================================================================
# 3. BERTSCORE
# =========================================================================
def compute_bertscore(df, cfg):
    from bert_score import score as bertscore
    scoreable = df[df["has_pair"] == True]
    P, R, F1 = bertscore(
        scoreable[cfg["MT_COL"]].tolist(), scoreable[cfg["REF_COL"]].tolist(),
        lang=TARGET_LANG_CODE, model_type=cfg["BERTSCORE_MODEL"],
        batch_size=cfg["BERTSCORE_BATCH_SIZE"], device=device, verbose=True
    )
    df.loc[scoreable.index, "bertscore_precision"] = P.numpy()
    df.loc[scoreable.index, "bertscore_recall"] = R.numpy()
    df.loc[scoreable.index, "bertscore_f1"] = F1.numpy()
    free_gpu()
    return df


# =========================================================================
# 4. LaBSE COSINE SIMILARITY
# =========================================================================
def compute_cosine_labse(df, cfg):
    from sentence_transformers import SentenceTransformer, util
    labse = SentenceTransformer("sentence-transformers/LaBSE", device=device)
    scoreable = df[df["has_pair"] == True]

    emb_mt = labse.encode(scoreable[cfg["MT_COL"]].tolist(), batch_size=32, show_progress_bar=True, convert_to_tensor=True)
    emb_ref = labse.encode(scoreable[cfg["REF_COL"]].tolist(), batch_size=32, show_progress_bar=True, convert_to_tensor=True)
    emb_src = labse.encode(scoreable[cfg["SOURCE_COL"]].tolist(), batch_size=32, show_progress_bar=True, convert_to_tensor=True)

    df.loc[scoreable.index, "cosine_mt_vs_ref"] = util.pairwise_cos_sim(emb_mt, emb_ref).cpu().numpy()
    df.loc[scoreable.index, "cosine_src_vs_ref_direct"] = util.pairwise_cos_sim(emb_src, emb_ref).cpu().numpy()
    del labse
    free_gpu()
    return df


# =========================================================================
# 5. COMET
# =========================================================================
def compute_comet(df, cfg):
    if not cfg["RUN_COMET"]:
        print("  Skipping COMET (CONFIG['RUN_COMET'] = False).")
        return df
    from comet import download_model, load_from_checkpoint
    scoreable = df[df["has_pair"] == True]
    print("Loading COMET (reference-based)...")
    path = download_model("Unbabel/wmt22-comet-da")
    model = load_from_checkpoint(path)
    data = [{"src": s, "mt": m, "ref": r} for s, m, r in
            zip(scoreable[cfg["SOURCE_COL"]], scoreable[cfg["MT_COL"]], scoreable[cfg["REF_COL"]])]
    out = model.predict(data, batch_size=8, gpus=1 if device == "cuda" else 0)
    df.loc[scoreable.index, "comet"] = out.scores
    del model
    free_gpu()
    return df


# =========================================================================
# 6. ENTITY / NUMERIC / STRUCTURAL CHECKS
# =========================================================================
NUM_RE = re.compile(r"\d+(?:\.\d+)?")
DEVANAGARI_RE = re.compile(r"[\u0900-\u097F]")
CAP_ENTITY_RE = re.compile(r"\b[A-Z][A-Za-z]{2,}\b")

def extract_numbers(text):
    return set(NUM_RE.findall(text))

def numeric_consistency(src, mt):
    src_nums = extract_numbers(src)
    if not src_nums:
        return 1.0
    return len(src_nums & extract_numbers(mt)) / len(src_nums)

def extract_capitalized_entities(text):
    return set(w.upper() for w in CAP_ENTITY_RE.findall(text))

def entity_preservation(ref, mt):
    ref_ents = extract_capitalized_entities(ref)
    if not ref_ents:
        return 1.0
    return len(ref_ents & extract_capitalized_entities(mt)) / len(ref_ents)

def has_untranslated_source_script(mt_text):
    return bool(DEVANAGARI_RE.search(mt_text))

def length_ratio(src, mt):
    src_len = max(len(src.split()), 1)
    return len(mt.split()) / src_len

def compute_structural_checks(df, cfg):
    df["numeric_consistency"] = df.apply(
        lambda r: numeric_consistency(r[cfg["SOURCE_COL"]], r[cfg["MT_COL"]]), axis=1)
    scoreable = df[df["has_pair"] == True]
    df.loc[scoreable.index, "entity_preservation"] = scoreable.apply(
        lambda r: entity_preservation(r[cfg["REF_COL"]], r[cfg["MT_COL"]]), axis=1)
    df["has_untranslated_text"] = df[cfg["MT_COL"]].apply(has_untranslated_source_script)
    df["length_ratio"] = df.apply(
        lambda r: length_ratio(r[cfg["SOURCE_COL"]], r[cfg["MT_COL"]]), axis=1)
    return df


# =========================================================================
# 7. FLAGS (per-row, no aggregate needs_review column)
# =========================================================================
def compute_flags(df, cfg):
    if "bertscore_f1" in df:
        df["flag_low_bertscore"] = df["bertscore_f1"] < cfg["THRESH_BERTSCORE_F1"]
    if "cosine_mt_vs_ref" in df:
        df["flag_low_cosine"] = df["cosine_mt_vs_ref"] < cfg["THRESH_COSINE"]
    if "comet" in df:
        df["flag_low_comet"] = df["comet"] < cfg["THRESH_COMET"]
    if "chrf++" in df:
        df["flag_low_chrf"] = df["chrf++"] < cfg["THRESH_CHRF"]
    if "has_untranslated_text" in df:
        df["flag_untranslated_text"] = df["has_untranslated_text"]
    if "numeric_consistency" in df:
        df["flag_low_numeric_consistency"] = df["numeric_consistency"] < cfg["THRESH_NUMERIC_CONSISTENCY"]
    if "entity_preservation" in df:
        df["flag_low_entity_preservation"] = df["entity_preservation"] < cfg["THRESH_ENTITY_PRESERVATION"]
    return df


# =========================================================================
# 8. FINAL REPORT: metric_accuracy_by_section.xlsx
# =========================================================================
def build_accuracy_report(df, cfg):
    scored = df[df["has_pair"] == True].copy()
    if len(scored) == 0:
        print("No scoreable rows -- skipping report.")
        return

    flagged_metrics = [
        ("chrf++", cfg["THRESH_CHRF"]),
        ("bertscore_f1", cfg["THRESH_BERTSCORE_F1"]),
        ("cosine_mt_vs_ref", cfg["THRESH_COSINE"]),
        ("numeric_consistency", cfg["THRESH_NUMERIC_CONSISTENCY"]),
        ("entity_preservation", cfg["THRESH_ENTITY_PRESERVATION"]),
    ]
    if cfg["RUN_COMET"]:
        flagged_metrics.insert(3, ("comet", cfg["THRESH_COMET"]))

    diagnostic_metrics = ["bleu", "ter", "rouge_l", "meteor", "cosine_src_vs_ref_direct", "length_ratio"]
    sections = sorted(scored["field_name"].unique())

    acc_rows = []
    for sec in sections:
        sub = scored[scored["field_name"] == sec]
        row = {"Section (Advisory)": sec, "n": len(sub)}
        for metric, thresh in flagged_metrics:
            if metric in sub.columns:
                row[metric] = round((sub[metric] >= thresh).mean() * 100, 1)
        acc_rows.append(row)
    acc_df = pd.DataFrame(acc_rows).set_index("Section (Advisory)")

    all_metric_cols = [m for m, _ in flagged_metrics] + diagnostic_metrics
    med_rows = []
    for sec in sections:
        sub = scored[scored["field_name"] == sec]
        row = {"Section (Advisory)": sec, "n": len(sub)}
        for metric in all_metric_cols:
            if metric in sub.columns:
                row[metric] = round(sub[metric].median(), 3)
        med_rows.append(row)
    med_df = pd.DataFrame(med_rows).set_index("Section (Advisory)")

    wb = Workbook()
    arial = "Arial"
    title_font = Font(name=arial, size=13, bold=True)
    header_font = Font(name=arial, size=10, bold=True)
    body_font = Font(name=arial, size=10)
    wrap = Alignment(wrap_text=True, vertical="top", horizontal="left")
    center = Alignment(wrap_text=True, vertical="top", horizontal="center")
    thin = Side(style="thin", color="000000")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)

    def write_table(ws, title, table_df, subtitle=""):
        ws.merge_cells(f"A1:{get_column_letter(len(table_df.columns)+1)}1")
        ws["A1"] = title
        ws["A1"].font = title_font
        ws.row_dimensions[1].height = 22
        r0 = 2
        if subtitle:
            ws.merge_cells(f"A2:{get_column_letter(len(table_df.columns)+1)}2")
            ws["A2"] = subtitle
            ws["A2"].font = Font(name=arial, size=9, italic=True)
            r0 = 3
        headers = ["Section (Advisory)"] + list(table_df.columns)
        for c, h in enumerate(headers, start=1):
            cell = ws.cell(row=r0, column=c, value=h)
            cell.font = header_font
            cell.border = border
            cell.alignment = center
        r = r0 + 1
        for idx, row in table_df.iterrows():
            ws.cell(row=r, column=1, value=idx).font = body_font
            ws.cell(row=r, column=1).border = border
            ws.cell(row=r, column=1).alignment = wrap
            for c, val in enumerate(row, start=2):
                cell = ws.cell(row=r, column=c, value=val)
                cell.font = body_font
                cell.border = border
                cell.alignment = center
            r += 1
        widths = [26] + [16] * len(table_df.columns)
        for i, w in enumerate(widths, start=1):
            ws.column_dimensions[get_column_letter(i)].width = w
        ws.freeze_panes = f"A{r0+1}"

    ws1 = wb.active
    ws1.title = "Accuracy % by Section"
    write_table(ws1, f"Accuracy % by Advisory Section ({SOURCE_LANG_NAME} -> English)",
                acc_df, "Each cell = % of that section's rows meeting the metric's accuracy threshold.")

    ws2 = wb.create_sheet("Median Scores by Section")
    write_table(ws2, "Median Raw Score by Advisory Section (per metric)",
                med_df, "Diagnostic metrics shown as median only -- not used for pass/fail.")

    ws3 = wb.create_sheet("Metric Definitions")
    ws3["A1"] = "Metric Definitions and Thresholds"
    ws3["A1"].font = title_font
    defs = [
        ("bleu", "0-100", "Diagnostic only", "Word n-gram overlap. Naturally low for paraphrastic MT."),
        ("chrf++", "0-100", f">= {cfg['THRESH_CHRF']}", "Character n-gram overlap; more forgiving than BLEU."),
        ("ter", "0-100+ (lower=better)", "Diagnostic only", "Edit distance to fix MT into reference."),
        ("rouge_l", "0-1", "Diagnostic only", "Longest common word-sequence overlap."),
        ("meteor", "0-1", "Diagnostic only", "Overlap with stemming/synonym matching."),
        ("bertscore_f1", "0-1", f">= {cfg['THRESH_BERTSCORE_F1']}", "Embedding-based semantic overlap."),
        ("cosine_mt_vs_ref", "0-1 (theoretical -1 to 1)", f">= {cfg['THRESH_COSINE']}", "LaBSE meaning-similarity, MT vs gold reference."),
        ("cosine_src_vs_ref_direct", "0-1 (theoretical -1 to 1)", "Diagnostic only (checks reference, not model)", f"LaBSE similarity, {SOURCE_LANG_NAME} source vs gold reference directly."),
        ("comet", "Roughly 0-1", f">= {cfg['THRESH_COMET']}" if cfg["RUN_COMET"] else "Skipped this run", "Trained quality-estimation model; closest to human judgment."),
        ("numeric_consistency", "0-1", f">= {cfg['THRESH_NUMERIC_CONSISTENCY']}", "Fraction of source numbers preserved in MT."),
        ("entity_preservation", "0-1", f">= {cfg['THRESH_ENTITY_PRESERVATION']}", "Fraction of reference's named entities found in MT."),
        ("length_ratio", "Roughly 0-2 (1.0=same length)", "Diagnostic only", "MT word count / source word count."),
    ]
    headers = ["Metric", "Range", "Accuracy Threshold", "What It Measures"]
    for c, h in enumerate(headers, start=1):
        cell = ws3.cell(row=3, column=c, value=h)
        cell.font = header_font
        cell.border = border
        cell.alignment = center
    r = 4
    for metric, scale, thresh, desc in defs:
        vals = [metric, scale, thresh, desc]
        for c, v in enumerate(vals, start=1):
            cell = ws3.cell(row=r, column=c, value=v)
            cell.font = body_font
            cell.border = border
            cell.alignment = wrap
        ws3.row_dimensions[r].height = 32
        r += 1
    widths = [22, 26, 30, 55]
    for i, w in enumerate(widths, start=1):
        ws3.column_dimensions[get_column_letter(i)].width = w
    ws3.freeze_panes = "A4"

    wb.save(cfg["REPORT_OUTPUT_FILE"])
    print(f"\nSaved {cfg['REPORT_OUTPUT_FILE']}")
    print("\n--- Accuracy % by Section ---")
    print(acc_df)


# =========================================================================
# MAIN
# =========================================================================
def run_validation_pipeline(cfg):
    overall_start = time.time()

    print("=== STEP 1: Merging gold references into translated data ===")
    df = merge_gold_references(cfg)

    print("\n=== STEP 2: N-gram / edit-distance metrics (diagnostic) ===")
    df = compute_ngram_metrics(df, cfg)

    print("\n=== STEP 3: BERTScore ===")
    df = compute_bertscore(df, cfg)

    print("\n=== STEP 4: LaBSE cosine similarity ===")
    df = compute_cosine_labse(df, cfg)

    print("\n=== STEP 5: COMET ===")
    df = compute_comet(df, cfg)

    print("\n=== STEP 6: Entity / numeric / structural checks ===")
    df = compute_structural_checks(df, cfg)

    print("\n=== STEP 7: Flags ===")
    df = compute_flags(df, cfg)

    df.to_excel(cfg["RESULTS_OUTPUT_FILE"], index=False)
    print(f"Saved {cfg['RESULTS_OUTPUT_FILE']}")

    print("\n=== STEP 8: Building per-section accuracy report ===")
    build_accuracy_report(df, cfg)

    print(f"\nAll done in {(time.time()-overall_start)/60:.1f} min total.")
    return df


# ---- RUN ----
final_df = run_validation_pipeline(CONFIG)

Using device: cuda
Mounted at /content/drive
Outputs will be saved to: /content/drive/MyDrive/agromet_pipeline_outputs
=== STEP 1: Merging gold references into translated data ===
Sections found in translated file: ['forecast_summary', 'bulletin_subtitle', 'weather_warnings', 'weather_warning_impacts', 'weather_impacts_general', 'general_advisory', 'sms_advisory', 'impact_based_advisories', 'crop_advisory', 'horticulture_advisory', 'livestock_advisory', 'fisheries_advisory', 'poultry_advisory', 'masthead', 'preamble', 'unmatched_other']

  forecast_summary                 rows=  559  paired=  520 (93.0%)
  bulletin_subtitle                rows=  560  paired=  523 (93.4%)
  weather_warnings                 rows=  180  paired=  168 (93.3%)
  weather_warning_impacts          rows=  180  paired=  168 (93.3%)
  weather_impacts_general          rows=  219  paired=  161 (73.5%)
  general_advisory                 rows=  560  paired=  523 (93.4%)
  sms_advisory                     rows=  560  p

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/729 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.62G [00:00<?, ?B/s]

calculating scores...
computing bert embedding.


  0%|          | 0/451 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 69.91 GiB. GPU 0 has a total capacity of 14.56 GiB of which 417.81 MiB is free. Including non-PyTorch memory, this process has 14.15 GiB memory in use. Of the allocated memory 13.98 GiB is allocated by PyTorch, and 50.91 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
# =========================================================================
# CELL 2 -- run this AFTER Cell 1 finished AND you restarted the runtime.
# Do not run Cell 1's pip installs again in this session -- they already
# persisted through the restart.
#
# FIX APPLIED: the previous OutOfMemoryError ("Tried to allocate 69.91 GiB")
# happened because at least one row has an extremely long mt_english /
# english_gold string. Transformer attention memory scales QUADRATICALLY
# with sequence length, so one oversized row can blow the GPU even with a
# small batch size. Fix: every transformer-based metric (BERTScore, LaBSE,
# COMET) now scores a length-capped COPY of the text (MAX_SCORING_WORDS),
# and each stage has automatic batch-size backoff + CPU fallback if it
# still hits OOM. The full, untruncated text is still what's saved in your
# output files -- only the copy fed into the scoring models is capped.
# =========================================================================

import os
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import re
import gc
import time
import pandas as pd
import numpy as np
import torch
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
if device == "cpu":
    print("No GPU detected -- will still run, just slower for BERTScore/COMET/LaBSE.")

# =========================================================================
# OPTIONAL: mount Google Drive so outputs survive a runtime reset
# =========================================================================
MOUNT_DRIVE = True
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/agromet_pipeline_outputs"

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    OUT_DIR = DRIVE_OUTPUT_DIR
else:
    OUT_DIR = "."
print(f"Outputs will be saved to: {OUT_DIR}")


# =========================================================================
# LANGUAGE CONFIG
# =========================================================================
SOURCE_LANG_NAME = "Hindi"
TARGET_LANG_CODE = "en"


# =========================================================================
# CONFIG
# =========================================================================
CONFIG = {
    "TRANSLATED_FILE": "translated_by_section.xlsx",
    "ALIGNED_FILE": "aligned_advisory_text.xlsx",
    "ALIGNED_SHEET": "advisory_paired",

    "RESULTS_OUTPUT_FILE": os.path.join(OUT_DIR, "validation_results_all_sections.xlsx"),
    "REPORT_OUTPUT_FILE": os.path.join(OUT_DIR, "metric_accuracy_by_section.xlsx"),
    "MERGED_FILE": os.path.join(OUT_DIR, "merged_with_gold_all_sections.xlsx"),

    "SOURCE_COL": "hindi_text",
    "MT_COL": "mt_english",
    "REF_COL": "english_gold",

    "JOIN_KEYS": ["state", "district", "month", "date"],

    "SECTION_MAPPING": {
        "forecast_summary": "forecast_summary",
        "bulletin_subtitle": "bulletin_subtitle",
        "weather_warnings": "weather_warnings",
        "weather_warning_impacts": "weather_warning_impacts",
        "weather_impacts_general": "weather_impacts_general",
        "general_advisory": "general_advisory",
        "sms_advisory": "sms_advisory",
        "impact_based_advisories": "impact_based_advisories_general",
        "crop_advisory": "crop_advisory",
        "horticulture_advisory": "horticulture_advisory",
        "livestock_advisory": "livestock_advisory",
        "fisheries_advisory": "fisheries_advisory",
        "poultry_advisory": "poultry_advisory",
        "masthead": "__masthead__",
        "preamble": "__preamble__",
        "unmatched_other": "unmatched_other",
    },

    "BERTSCORE_MODEL": "microsoft/deberta-large-mnli",
    # Lowered from 8 -- the batch size wasn't actually the problem (a single
    # oversized sequence was), but a smaller batch gives extra headroom.
    "BERTSCORE_BATCH_SIZE": 4 if device == "cuda" else 2,

    "RUN_COMET": True,
    "COMET_BATCH_SIZE": 4 if device == "cuda" else 2,
    "LABSE_BATCH_SIZE": 16 if device == "cuda" else 8,

    # NEW: hard cap on words fed into any transformer-based metric. This is
    # what actually fixes the OOM -- attention memory scales with the
    # SQUARE of sequence length, so one very long row can blow 14GB even
    # solo. 400 words is generous for a forecast/advisory sentence or two;
    # raise it only if you confirm your GPU has headroom.
    "MAX_SCORING_WORDS": 400,

    "THRESH_BERTSCORE_F1": 0.80,
    "THRESH_COSINE": 0.75,
    "THRESH_COMET": 0.75,
    "THRESH_CHRF": 40.0,
    "THRESH_NUMERIC_CONSISTENCY": 0.5,
    "THRESH_ENTITY_PRESERVATION": 0.4,
}

if device == "cpu" and CONFIG["RUN_COMET"]:
    print("NOTE: COMET on CPU is slow across ~5000 rows. Set CONFIG['RUN_COMET'] = False "
          "above if you want to skip it and rely on BERTScore + LaBSE cosine instead.")


# =========================================================================
# NEW: length-safety helper -- used before every transformer-based metric
# =========================================================================
def safe_truncate(text, max_words):
    text = str(text)
    words = text.split()
    if len(words) <= max_words:
        return text
    return " ".join(words[:max_words])


# =========================================================================
# 1. MERGE
# =========================================================================
def merge_gold_references(cfg):
    aligned = pd.read_excel(cfg["ALIGNED_FILE"], sheet_name=cfg["ALIGNED_SHEET"])
    xl = pd.ExcelFile(cfg["TRANSLATED_FILE"])

    frames = []
    print(f"Sections found in translated file: {xl.sheet_names}\n")
    for sheet in xl.sheet_names:
        prefix = cfg["SECTION_MAPPING"].get(sheet)
        if prefix is None:
            print(f"  Skipping '{sheet}': no mapping to an aligned-file column.")
            continue
        en_col = f"{prefix} [EN]"
        if en_col not in aligned.columns:
            print(f"  Skipping '{sheet}': column '{en_col}' not found in aligned file.")
            continue

        df = xl.parse(sheet)
        gold_sub = aligned[cfg["JOIN_KEYS"] + [en_col]].rename(columns={en_col: cfg["REF_COL"]})
        gold_sub = gold_sub.drop_duplicates(subset=cfg["JOIN_KEYS"], keep="first")

        merged = df.merge(gold_sub, on=cfg["JOIN_KEYS"], how="left")
        merged[cfg["REF_COL"]] = merged[cfg["REF_COL"]].astype(str)

        cleaned = merged[cfg["REF_COL"]].str.strip()
        merged["has_pair"] = ~cleaned.isin(["", "nan", "None", "NaN"])

        merged["field_name"] = sheet
        merged["sheet"] = sheet

        n_paired = merged["has_pair"].sum()
        print(f"  {sheet:32s} rows={len(merged):5d}  paired={n_paired:5d} ({n_paired/len(merged)*100:.1f}%)")
        frames.append(merged)

    full = pd.concat(frames, ignore_index=True)
    full[cfg["SOURCE_COL"]] = full[cfg["SOURCE_COL"]].astype(str)
    full[cfg["MT_COL"]] = full[cfg["MT_COL"]].astype(str)

    with pd.ExcelWriter(cfg["MERGED_FILE"], engine="openpyxl") as writer:
        for sheet in full["field_name"].unique():
            full[full["field_name"] == sheet].to_excel(writer, sheet_name=sheet[:31], index=False)
    print(f"\nSaved {cfg['MERGED_FILE']}")

    total_paired = full["has_pair"].sum()
    print(f"\nTotal rows: {len(full)}  |  Total scoreable (paired) rows: {total_paired} "
          f"({total_paired/len(full)*100:.1f}%)")

    # NEW: diagnostic -- show the longest rows so you know what triggered
    # the OOM and can sanity-check whether truncation is actually safe for it
    scoreable = full[full["has_pair"] == True]
    mt_wc = scoreable[cfg["MT_COL"]].str.split().str.len()
    ref_wc = scoreable[cfg["REF_COL"]].str.split().str.len()
    print(f"\nLongest mt_english row: {mt_wc.max()} words "
          f"(section: {scoreable.loc[mt_wc.idxmax(), 'field_name']})")
    print(f"Longest english_gold row: {ref_wc.max()} words "
          f"(section: {scoreable.loc[ref_wc.idxmax(), 'field_name']})")
    n_over_cap = ((mt_wc > cfg["MAX_SCORING_WORDS"]) | (ref_wc > cfg["MAX_SCORING_WORDS"])).sum()
    print(f"Rows exceeding MAX_SCORING_WORDS={cfg['MAX_SCORING_WORDS']}: {n_over_cap} "
          f"(these will be truncated for scoring only, not in your saved output text)")

    return full


# =========================================================================
# 2. N-GRAM / EDIT-DISTANCE METRICS (diagnostic only)
# =========================================================================
def compute_ngram_metrics(df, cfg):
    import sacrebleu
    from rouge_score import rouge_scorer
    import nltk
    nltk.download("wordnet", quiet=True)
    nltk.download("omw-1.4", quiet=True)
    nltk.download("punkt", quiet=True)
    nltk.download("punkt_tab", quiet=True)
    from nltk.translate.meteor_score import meteor_score

    rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    scoreable = df[df["has_pair"] == True]
    print(f"  Scoring {len(scoreable)} rows (BLEU/chrF++/TER/ROUGE-L/METEOR)...")

    bleu, chrf, ter, rougeL, meteor = {}, {}, {}, {}, {}
    start = time.time()
    for n, (idx, row) in enumerate(scoreable.iterrows(), 1):
        mt, ref = row[cfg["MT_COL"]], row[cfg["REF_COL"]]
        bleu[idx] = sacrebleu.sentence_bleu(mt, [ref]).score
        chrf[idx] = sacrebleu.sentence_chrf(mt, [ref], word_order=2).score
        ter[idx] = sacrebleu.sentence_ter(mt, [ref]).score
        rougeL[idx] = rouge.score(ref, mt)["rougeL"].fmeasure
        try:
            meteor[idx] = meteor_score([ref.split()], mt.split())
        except Exception:
            meteor[idx] = np.nan
        if n % 200 == 0 or n == len(scoreable):
            rate = n / (time.time() - start) if time.time() > start else 0
            print(f"    {n}/{len(scoreable)}  ({rate:.1f} rows/sec)")

    df["bleu"] = df.index.map(bleu)
    df["chrf++"] = df.index.map(chrf)
    df["ter"] = df.index.map(ter)
    df["rouge_l"] = df.index.map(rougeL)
    df["meteor"] = df.index.map(meteor)
    return df


# =========================================================================
# 3. BERTSCORE -- now with length capping + OOM backoff + CPU fallback
# =========================================================================
def compute_bertscore(df, cfg):
    from bert_score import score as bertscore
    scoreable = df[df["has_pair"] == True]

    mt_texts = [safe_truncate(t, cfg["MAX_SCORING_WORDS"]) for t in scoreable[cfg["MT_COL"]]]
    ref_texts = [safe_truncate(t, cfg["MAX_SCORING_WORDS"]) for t in scoreable[cfg["REF_COL"]]]

    batch_size = cfg["BERTSCORE_BATCH_SIZE"]
    run_device = device
    while True:
        try:
            free_gpu()
            P, R, F1 = bertscore(
                mt_texts, ref_texts,
                lang=TARGET_LANG_CODE, model_type=cfg["BERTSCORE_MODEL"],
                batch_size=batch_size, device=run_device, verbose=True
            )
            break
        except torch.cuda.OutOfMemoryError:
            free_gpu()
            if batch_size > 1:
                batch_size = max(1, batch_size // 2)
                print(f"CUDA OOM in BERTScore -- retrying with batch_size={batch_size}")
                continue
            if run_device == "cuda":
                print("Still OOM at batch_size=1 on GPU -- falling back to CPU for BERTScore "
                      "(slower, but will not crash).")
                run_device = "cpu"
                batch_size = 4
                continue
            raise  # already on CPU and still failing -- something else is wrong

    df.loc[scoreable.index, "bertscore_precision"] = P.numpy()
    df.loc[scoreable.index, "bertscore_recall"] = R.numpy()
    df.loc[scoreable.index, "bertscore_f1"] = F1.numpy()
    free_gpu()
    return df


# =========================================================================
# 4. LaBSE COSINE SIMILARITY -- now with length capping + OOM backoff
# =========================================================================
def compute_cosine_labse(df, cfg):
    from sentence_transformers import SentenceTransformer, util
    labse = SentenceTransformer("sentence-transformers/LaBSE", device=device)
    scoreable = df[df["has_pair"] == True]

    mt_texts = [safe_truncate(t, cfg["MAX_SCORING_WORDS"]) for t in scoreable[cfg["MT_COL"]]]
    ref_texts = [safe_truncate(t, cfg["MAX_SCORING_WORDS"]) for t in scoreable[cfg["REF_COL"]]]
    src_texts = [safe_truncate(t, cfg["MAX_SCORING_WORDS"]) for t in scoreable[cfg["SOURCE_COL"]]]

    def encode_safely(texts, batch_size):
        bs = batch_size
        while True:
            try:
                free_gpu()
                return labse.encode(texts, batch_size=bs, show_progress_bar=True, convert_to_tensor=True)
            except torch.cuda.OutOfMemoryError:
                free_gpu()
                if bs > 1:
                    bs = max(1, bs // 2)
                    print(f"CUDA OOM in LaBSE encode -- retrying with batch_size={bs}")
                    continue
                raise

    emb_mt = encode_safely(mt_texts, cfg["LABSE_BATCH_SIZE"])
    emb_ref = encode_safely(ref_texts, cfg["LABSE_BATCH_SIZE"])
    emb_src = encode_safely(src_texts, cfg["LABSE_BATCH_SIZE"])

    df.loc[scoreable.index, "cosine_mt_vs_ref"] = util.pairwise_cos_sim(emb_mt, emb_ref).cpu().numpy()
    df.loc[scoreable.index, "cosine_src_vs_ref_direct"] = util.pairwise_cos_sim(emb_src, emb_ref).cpu().numpy()
    del labse
    free_gpu()
    return df


# =========================================================================
# 5. COMET -- now with length capping + OOM backoff + CPU fallback
# =========================================================================
def compute_comet(df, cfg):
    if not cfg["RUN_COMET"]:
        print("  Skipping COMET (CONFIG['RUN_COMET'] = False).")
        return df
    from comet import download_model, load_from_checkpoint
    scoreable = df[df["has_pair"] == True]
    print("Loading COMET (reference-based)...")
    path = download_model("Unbabel/wmt22-comet-da")
    model = load_from_checkpoint(path)

    data = [
        {
            "src": safe_truncate(s, cfg["MAX_SCORING_WORDS"]),
            "mt": safe_truncate(m, cfg["MAX_SCORING_WORDS"]),
            "ref": safe_truncate(r, cfg["MAX_SCORING_WORDS"]),
        }
        for s, m, r in zip(scoreable[cfg["SOURCE_COL"]], scoreable[cfg["MT_COL"]], scoreable[cfg["REF_COL"]])
    ]

    batch_size = cfg["COMET_BATCH_SIZE"]
    gpus = 1 if device == "cuda" else 0
    while True:
        try:
            free_gpu()
            out = model.predict(data, batch_size=batch_size, gpus=gpus)
            break
        except torch.cuda.OutOfMemoryError:
            free_gpu()
            if batch_size > 1:
                batch_size = max(1, batch_size // 2)
                print(f"CUDA OOM in COMET -- retrying with batch_size={batch_size}")
                continue
            if gpus == 1:
                print("Still OOM at batch_size=1 on GPU -- falling back to CPU for COMET "
                      "(slower, but will not crash).")
                gpus = 0
                batch_size = 4
                continue
            raise

    df.loc[scoreable.index, "comet"] = out.scores
    del model
    free_gpu()
    return df


# =========================================================================
# 6. ENTITY / NUMERIC / STRUCTURAL CHECKS  (unchanged -- pure regex, no GPU)
# =========================================================================
NUM_RE = re.compile(r"\d+(?:\.\d+)?")
DEVANAGARI_RE = re.compile(r"[\u0900-\u097F]")
CAP_ENTITY_RE = re.compile(r"\b[A-Z][A-Za-z]{2,}\b")

def extract_numbers(text):
    return set(NUM_RE.findall(text))

def numeric_consistency(src, mt):
    src_nums = extract_numbers(src)
    if not src_nums:
        return 1.0
    return len(src_nums & extract_numbers(mt)) / len(src_nums)

def extract_capitalized_entities(text):
    return set(w.upper() for w in CAP_ENTITY_RE.findall(text))

def entity_preservation(ref, mt):
    ref_ents = extract_capitalized_entities(ref)
    if not ref_ents:
        return 1.0
    return len(ref_ents & extract_capitalized_entities(mt)) / len(ref_ents)

def has_untranslated_source_script(mt_text):
    return bool(DEVANAGARI_RE.search(mt_text))

def length_ratio(src, mt):
    src_len = max(len(src.split()), 1)
    return len(mt.split()) / src_len

def compute_structural_checks(df, cfg):
    df["numeric_consistency"] = df.apply(
        lambda r: numeric_consistency(r[cfg["SOURCE_COL"]], r[cfg["MT_COL"]]), axis=1)
    scoreable = df[df["has_pair"] == True]
    df.loc[scoreable.index, "entity_preservation"] = scoreable.apply(
        lambda r: entity_preservation(r[cfg["REF_COL"]], r[cfg["MT_COL"]]), axis=1)
    df["has_untranslated_text"] = df[cfg["MT_COL"]].apply(has_untranslated_source_script)
    df["length_ratio"] = df.apply(
        lambda r: length_ratio(r[cfg["SOURCE_COL"]], r[cfg["MT_COL"]]), axis=1)
    return df


# =========================================================================
# 7. FLAGS
# =========================================================================
def compute_flags(df, cfg):
    if "bertscore_f1" in df:
        df["flag_low_bertscore"] = df["bertscore_f1"] < cfg["THRESH_BERTSCORE_F1"]
    if "cosine_mt_vs_ref" in df:
        df["flag_low_cosine"] = df["cosine_mt_vs_ref"] < cfg["THRESH_COSINE"]
    if "comet" in df:
        df["flag_low_comet"] = df["comet"] < cfg["THRESH_COMET"]
    if "chrf++" in df:
        df["flag_low_chrf"] = df["chrf++"] < cfg["THRESH_CHRF"]
    if "has_untranslated_text" in df:
        df["flag_untranslated_text"] = df["has_untranslated_text"]
    if "numeric_consistency" in df:
        df["flag_low_numeric_consistency"] = df["numeric_consistency"] < cfg["THRESH_NUMERIC_CONSISTENCY"]
    if "entity_preservation" in df:
        df["flag_low_entity_preservation"] = df["entity_preservation"] < cfg["THRESH_ENTITY_PRESERVATION"]
    return df


# =========================================================================
# 8. FINAL REPORT: metric_accuracy_by_section.xlsx  (unchanged)
# =========================================================================
def build_accuracy_report(df, cfg):
    scored = df[df["has_pair"] == True].copy()
    if len(scored) == 0:
        print("No scoreable rows -- skipping report.")
        return

    flagged_metrics = [
        ("chrf++", cfg["THRESH_CHRF"]),
        ("bertscore_f1", cfg["THRESH_BERTSCORE_F1"]),
        ("cosine_mt_vs_ref", cfg["THRESH_COSINE"]),
        ("numeric_consistency", cfg["THRESH_NUMERIC_CONSISTENCY"]),
        ("entity_preservation", cfg["THRESH_ENTITY_PRESERVATION"]),
    ]
    if cfg["RUN_COMET"]:
        flagged_metrics.insert(3, ("comet", cfg["THRESH_COMET"]))

    diagnostic_metrics = ["bleu", "ter", "rouge_l", "meteor", "cosine_src_vs_ref_direct", "length_ratio"]
    sections = sorted(scored["field_name"].unique())

    acc_rows = []
    for sec in sections:
        sub = scored[scored["field_name"] == sec]
        row = {"Section (Advisory)": sec, "n": len(sub)}
        for metric, thresh in flagged_metrics:
            if metric in sub.columns:
                row[metric] = round((sub[metric] >= thresh).mean() * 100, 1)
        acc_rows.append(row)
    acc_df = pd.DataFrame(acc_rows).set_index("Section (Advisory)")

    all_metric_cols = [m for m, _ in flagged_metrics] + diagnostic_metrics
    med_rows = []
    for sec in sections:
        sub = scored[scored["field_name"] == sec]
        row = {"Section (Advisory)": sec, "n": len(sub)}
        for metric in all_metric_cols:
            if metric in sub.columns:
                row[metric] = round(sub[metric].median(), 3)
        med_rows.append(row)
    med_df = pd.DataFrame(med_rows).set_index("Section (Advisory)")

    wb = Workbook()
    arial = "Arial"
    title_font = Font(name=arial, size=13, bold=True)
    header_font = Font(name=arial, size=10, bold=True)
    body_font = Font(name=arial, size=10)
    wrap = Alignment(wrap_text=True, vertical="top", horizontal="left")
    center = Alignment(wrap_text=True, vertical="top", horizontal="center")
    thin = Side(style="thin", color="000000")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)

    def write_table(ws, title, table_df, subtitle=""):
        ws.merge_cells(f"A1:{get_column_letter(len(table_df.columns)+1)}1")
        ws["A1"] = title
        ws["A1"].font = title_font
        ws.row_dimensions[1].height = 22
        r0 = 2
        if subtitle:
            ws.merge_cells(f"A2:{get_column_letter(len(table_df.columns)+1)}2")
            ws["A2"] = subtitle
            ws["A2"].font = Font(name=arial, size=9, italic=True)
            r0 = 3
        headers = ["Section (Advisory)"] + list(table_df.columns)
        for c, h in enumerate(headers, start=1):
            cell = ws.cell(row=r0, column=c, value=h)
            cell.font = header_font
            cell.border = border
            cell.alignment = center
        r = r0 + 1
        for idx, row in table_df.iterrows():
            ws.cell(row=r, column=1, value=idx).font = body_font
            ws.cell(row=r, column=1).border = border
            ws.cell(row=r, column=1).alignment = wrap
            for c, val in enumerate(row, start=2):
                cell = ws.cell(row=r, column=c, value=val)
                cell.font = body_font
                cell.border = border
                cell.alignment = center
            r += 1
        widths = [26] + [16] * len(table_df.columns)
        for i, w in enumerate(widths, start=1):
            ws.column_dimensions[get_column_letter(i)].width = w
        ws.freeze_panes = f"A{r0+1}"

    ws1 = wb.active
    ws1.title = "Accuracy % by Section"
    write_table(ws1, f"Accuracy % by Advisory Section ({SOURCE_LANG_NAME} -> English)",
                acc_df, "Each cell = % of that section's rows meeting the metric's accuracy threshold.")

    ws2 = wb.create_sheet("Median Scores by Section")
    write_table(ws2, "Median Raw Score by Advisory Section (per metric)",
                med_df, "Diagnostic metrics shown as median only -- not used for pass/fail.")

    ws3 = wb.create_sheet("Metric Definitions")
    ws3["A1"] = "Metric Definitions and Thresholds"
    ws3["A1"].font = title_font
    defs = [
        ("bleu", "0-100", "Diagnostic only", "Word n-gram overlap. Naturally low for paraphrastic MT."),
        ("chrf++", "0-100", f">= {cfg['THRESH_CHRF']}", "Character n-gram overlap; more forgiving than BLEU."),
        ("ter", "0-100+ (lower=better)", "Diagnostic only", "Edit distance to fix MT into reference."),
        ("rouge_l", "0-1", "Diagnostic only", "Longest common word-sequence overlap."),
        ("meteor", "0-1", "Diagnostic only", "Overlap with stemming/synonym matching."),
        ("bertscore_f1", "0-1", f">= {cfg['THRESH_BERTSCORE_F1']}", "Embedding-based semantic overlap."),
        ("cosine_mt_vs_ref", "0-1 (theoretical -1 to 1)", f">= {cfg['THRESH_COSINE']}", "LaBSE meaning-similarity, MT vs gold reference."),
        ("cosine_src_vs_ref_direct", "0-1 (theoretical -1 to 1)", "Diagnostic only (checks reference, not model)", f"LaBSE similarity, {SOURCE_LANG_NAME} source vs gold reference directly."),
        ("comet", "Roughly 0-1", f">= {cfg['THRESH_COMET']}" if cfg["RUN_COMET"] else "Skipped this run", "Trained quality-estimation model; closest to human judgment."),
        ("numeric_consistency", "0-1", f">= {cfg['THRESH_NUMERIC_CONSISTENCY']}", "Fraction of source numbers preserved in MT."),
        ("entity_preservation", "0-1", f">= {cfg['THRESH_ENTITY_PRESERVATION']}", "Fraction of reference's named entities found in MT."),
        ("length_ratio", "Roughly 0-2 (1.0=same length)", "Diagnostic only", "MT word count / source word count."),
    ]
    headers = ["Metric", "Range", "Accuracy Threshold", "What It Measures"]
    for c, h in enumerate(headers, start=1):
        cell = ws3.cell(row=3, column=c, value=h)
        cell.font = header_font
        cell.border = border
        cell.alignment = center
    r = 4
    for metric, scale, thresh, desc in defs:
        vals = [metric, scale, thresh, desc]
        for c, v in enumerate(vals, start=1):
            cell = ws3.cell(row=r, column=c, value=v)
            cell.font = body_font
            cell.border = border
            cell.alignment = wrap
        ws3.row_dimensions[r].height = 32
        r += 1
    widths = [22, 26, 30, 55]
    for i, w in enumerate(widths, start=1):
        ws3.column_dimensions[get_column_letter(i)].width = w
    ws3.freeze_panes = "A4"

    wb.save(cfg["REPORT_OUTPUT_FILE"])
    print(f"\nSaved {cfg['REPORT_OUTPUT_FILE']}")
    print("\n--- Accuracy % by Section ---")
    print(acc_df)


# =========================================================================
# MAIN
# =========================================================================
def run_validation_pipeline(cfg):
    overall_start = time.time()

    print("=== STEP 1: Merging gold references into translated data ===")
    df = merge_gold_references(cfg)

    print("\n=== STEP 2: N-gram / edit-distance metrics (diagnostic) ===")
    df = compute_ngram_metrics(df, cfg)

    print("\n=== STEP 3: BERTScore ===")
    df = compute_bertscore(df, cfg)

    print("\n=== STEP 4: LaBSE cosine similarity ===")
    df = compute_cosine_labse(df, cfg)

    print("\n=== STEP 5: COMET ===")
    df = compute_comet(df, cfg)

    print("\n=== STEP 6: Entity / numeric / structural checks ===")
    df = compute_structural_checks(df, cfg)

    print("\n=== STEP 7: Flags ===")
    df = compute_flags(df, cfg)

    df.to_excel(cfg["RESULTS_OUTPUT_FILE"], index=False)
    print(f"Saved {cfg['RESULTS_OUTPUT_FILE']}")

    print("\n=== STEP 8: Building per-section accuracy report ===")
    build_accuracy_report(df, cfg)

    print(f"\nAll done in {(time.time()-overall_start)/60:.1f} min total.")
    return df


# ---- RUN ----
final_df = run_validation_pipeline(CONFIG)

Using device: cuda
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Outputs will be saved to: /content/drive/MyDrive/agromet_pipeline_outputs
=== STEP 1: Merging gold references into translated data ===
Sections found in translated file: ['forecast_summary', 'bulletin_subtitle', 'weather_warnings', 'weather_warning_impacts', 'weather_impacts_general', 'general_advisory', 'sms_advisory', 'impact_based_advisories', 'crop_advisory', 'horticulture_advisory', 'livestock_advisory', 'fisheries_advisory', 'poultry_advisory', 'masthead', 'preamble', 'unmatched_other']

  forecast_summary                 rows=  559  paired=  520 (93.0%)
  bulletin_subtitle                rows=  560  paired=  523 (93.4%)
  weather_warnings                 rows=  180  paired=  168 (93.3%)
  weather_warning_impacts          rows=  180  paired=  168 (93.3%)
  weather_impacts_general          rows=  219  paired=  161 (73.5%)
  general_adv

  0%|          | 0/901 [00:00<?, ?it/s]

In [1]:
# =========================================================================
# CELL 1 -- RUN THIS FIRST, THEN RESTART THE RUNTIME
# (unchanged from before -- skip if you already ran it this session)
# =========================================================================
!pip install -q --no-deps unbabel-comet
!pip install -q "pytorch-lightning>=2.0" "jsonargparse[signatures]>=4.27.5" "omegaconf>=2.3.0"
!pip install -q transformers sentencepiece accelerate sacrebleu rouge-score nltk bert-score sentence-transformers openpyxl

import numpy
print("\nnumpy version after install:", numpy.__version__)
if numpy.__version__.startswith("1."):
    print("WARNING: numpy downgraded. Run: !pip install -q --upgrade --no-deps \"numpy>=2\"  then restart.")
else:
    print("Good -- numpy is 2.x. Now: Runtime > Restart session, then run Cell 2.")


numpy version after install: 1.26.4


In [2]:
# =========================================================================
# CELL 2 -- run AFTER Cell 1 finished AND you restarted the runtime.
#
# KEY FIXES IN THIS VERSION:
#  1. RAM crash fix: processes ONE SECTION AT A TIME, writes a checkpoint
#     file after each, frees memory between sections. Peak RAM is bounded
#     to ~one section's data instead of all 6,089 rows at once.
#  2. GPU OOM fix: BERTScore/COMET truncate by ACTUAL MODEL TOKENS (not
#     word-split, which silently let a no-space/mangled row through last
#     time). No CPU fallback -- if GPU can't handle it, it fails fast
#     instead of quietly eating all system RAM for hours.
#  3. Speed fix: BERTScore, LaBSE, and COMET models are loaded ONCE up
#     front and reused across all 16 sections, instead of being reloaded
#     from scratch for every section (this alone should save a lot of time).
#  4. If Colab crashes partway through, just rerun this cell -- sections
#     that already have a checkpoint file are automatically skipped.
# =========================================================================

import os
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import re
import gc
import time
import pandas as pd
import numpy as np
import torch
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# =========================================================================
# Mount Drive
# =========================================================================
MOUNT_DRIVE = True
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/agromet_pipeline_outputs"

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    OUT_DIR = DRIVE_OUTPUT_DIR
else:
    OUT_DIR = "."
CHECKPOINT_DIR = os.path.join(OUT_DIR, "section_checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Outputs -> {OUT_DIR}")
print(f"Checkpoints -> {CHECKPOINT_DIR}")

SOURCE_LANG_NAME = "Hindi"
TARGET_LANG_CODE = "en"

CONFIG = {
    "TRANSLATED_FILE": "translated_by_section.xlsx",
    "ALIGNED_FILE": "aligned_advisory_text.xlsx",
    "ALIGNED_SHEET": "advisory_paired",

    "RESULTS_OUTPUT_FILE": os.path.join(OUT_DIR, "validation_results_all_sections.xlsx"),
    "REPORT_OUTPUT_FILE": os.path.join(OUT_DIR, "metric_accuracy_by_section.xlsx"),
    "MERGED_FILE": os.path.join(OUT_DIR, "merged_with_gold_all_sections.xlsx"),

    "SOURCE_COL": "hindi_text",
    "MT_COL": "mt_english",
    "REF_COL": "english_gold",
    "JOIN_KEYS": ["state", "district", "month", "date"],

    "SECTION_MAPPING": {
        "forecast_summary": "forecast_summary",
        "bulletin_subtitle": "bulletin_subtitle",
        "weather_warnings": "weather_warnings",
        "weather_warning_impacts": "weather_warning_impacts",
        "weather_impacts_general": "weather_impacts_general",
        "general_advisory": "general_advisory",
        "sms_advisory": "sms_advisory",
        "impact_based_advisories": "impact_based_advisories_general",
        "crop_advisory": "crop_advisory",
        "horticulture_advisory": "horticulture_advisory",
        "livestock_advisory": "livestock_advisory",
        "fisheries_advisory": "fisheries_advisory",
        "poultry_advisory": "poultry_advisory",
        "masthead": "__masthead__",
        "preamble": "__preamble__",
        "unmatched_other": "unmatched_other",
    },

    "BERTSCORE_MODEL": "microsoft/deberta-large-mnli",
    "BERTSCORE_BATCH_SIZE": 4 if device == "cuda" else 2,
    "RUN_COMET": True,
    "COMET_BATCH_SIZE": 4 if device == "cuda" else 2,
    "LABSE_BATCH_SIZE": 16 if device == "cuda" else 8,

    "MAX_TOKENS": 150,  # hard model-token cap -- fixes both the GPU OOM and the RAM crash

    "THRESH_BERTSCORE_F1": 0.80,
    "THRESH_COSINE": 0.75,
    "THRESH_COMET": 0.75,
    "THRESH_CHRF": 40.0,
    "THRESH_NUMERIC_CONSISTENCY": 0.5,
    "THRESH_ENTITY_PRESERVATION": 0.4,
}

if device == "cpu" and CONFIG["RUN_COMET"]:
    print("NOTE: COMET on CPU is slow. Set CONFIG['RUN_COMET']=False to skip it if needed.")


# =========================================================================
# Hard token-level truncation (the actual OOM/RAM-crash fix)
# =========================================================================
from transformers import AutoTokenizer

print("Loading tokenizers for truncation...")
_bertscore_tok = AutoTokenizer.from_pretrained(CONFIG["BERTSCORE_MODEL"])
_comet_tok = AutoTokenizer.from_pretrained("xlm-roberta-large")  # COMET's backbone tokenizer

def hard_truncate(text, tokenizer, max_tokens):
    text = str(text)
    if not text.strip():
        return text
    ids = tokenizer.encode(text, add_special_tokens=False, truncation=True, max_length=max_tokens)
    return tokenizer.decode(ids, skip_special_tokens=True)


# =========================================================================
# 1. MERGE (unchanged logic, same diagnostics)
# =========================================================================
def merge_gold_references(cfg):
    aligned = pd.read_excel(cfg["ALIGNED_FILE"], sheet_name=cfg["ALIGNED_SHEET"])
    xl = pd.ExcelFile(cfg["TRANSLATED_FILE"])

    frames = []
    print(f"Sections found: {xl.sheet_names}\n")
    for sheet in xl.sheet_names:
        prefix = cfg["SECTION_MAPPING"].get(sheet)
        if prefix is None:
            print(f"  Skipping '{sheet}': no mapping.")
            continue
        en_col = f"{prefix} [EN]"
        if en_col not in aligned.columns:
            print(f"  Skipping '{sheet}': column '{en_col}' not found.")
            continue

        df = xl.parse(sheet)
        gold_sub = aligned[cfg["JOIN_KEYS"] + [en_col]].rename(columns={en_col: cfg["REF_COL"]})
        gold_sub = gold_sub.drop_duplicates(subset=cfg["JOIN_KEYS"], keep="first")

        merged = df.merge(gold_sub, on=cfg["JOIN_KEYS"], how="left")
        merged[cfg["REF_COL"]] = merged[cfg["REF_COL"]].astype(str)
        cleaned = merged[cfg["REF_COL"]].str.strip()
        merged["has_pair"] = ~cleaned.isin(["", "nan", "None", "NaN"])
        merged["field_name"] = sheet
        merged["sheet"] = sheet

        n_paired = merged["has_pair"].sum()
        print(f"  {sheet:32s} rows={len(merged):5d}  paired={n_paired:5d} ({n_paired/len(merged)*100:.1f}%)")
        frames.append(merged)

    full = pd.concat(frames, ignore_index=True)
    full[cfg["SOURCE_COL"]] = full[cfg["SOURCE_COL"]].astype(str)
    full[cfg["MT_COL"]] = full[cfg["MT_COL"]].astype(str)

    with pd.ExcelWriter(cfg["MERGED_FILE"], engine="openpyxl") as writer:
        for sheet in full["field_name"].unique():
            full[full["field_name"] == sheet].to_excel(writer, sheet_name=sheet[:31], index=False)
    print(f"\nSaved {cfg['MERGED_FILE']}")

    total_paired = full["has_pair"].sum()
    print(f"Total rows: {len(full)} | Scoreable: {total_paired} ({total_paired/len(full)*100:.1f}%)")

    # diagnostic: find the runaway row using CHARACTER length (not word-split,
    # which is exactly what missed the offending row last time)
    scoreable = full[full["has_pair"] == True]
    mt_len = scoreable[cfg["MT_COL"]].str.len()
    print(f"\nLongest mt_english row: {mt_len.max()} characters "
          f"(section: {scoreable.loc[mt_len.idxmax(), 'field_name']})")

    return full


# =========================================================================
# 2. N-GRAM METRICS (unchanged, CPU-only, safe)
# =========================================================================
def compute_ngram_metrics(df, cfg):
    import sacrebleu
    from rouge_score import rouge_scorer
    import nltk
    nltk.download("wordnet", quiet=True)
    nltk.download("omw-1.4", quiet=True)
    nltk.download("punkt", quiet=True)
    nltk.download("punkt_tab", quiet=True)
    from nltk.translate.meteor_score import meteor_score

    rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    scoreable = df[df["has_pair"] == True]

    bleu, chrf, ter, rougeL, meteor = {}, {}, {}, {}, {}
    for idx, row in scoreable.iterrows():
        mt, ref = row[cfg["MT_COL"]], row[cfg["REF_COL"]]
        bleu[idx] = sacrebleu.sentence_bleu(mt, [ref]).score
        chrf[idx] = sacrebleu.sentence_chrf(mt, [ref], word_order=2).score
        ter[idx] = sacrebleu.sentence_ter(mt, [ref]).score
        rougeL[idx] = rouge.score(ref, mt)["rougeL"].fmeasure
        try:
            meteor[idx] = meteor_score([ref.split()], mt.split())
        except Exception:
            meteor[idx] = np.nan

    df["bleu"] = df.index.map(bleu)
    df["chrf++"] = df.index.map(chrf)
    df["ter"] = df.index.map(ter)
    df["rouge_l"] = df.index.map(rougeL)
    df["meteor"] = df.index.map(meteor)
    return df


# =========================================================================
# 3. BERTSCORE -- reuses a persistent scorer object, hard token truncation
# =========================================================================
def compute_bertscore(df, cfg, scorer):
    scoreable = df[df["has_pair"] == True]
    if len(scoreable) == 0:
        return df
    mt_texts = [hard_truncate(t, _bertscore_tok, cfg["MAX_TOKENS"]) for t in scoreable[cfg["MT_COL"]]]
    ref_texts = [hard_truncate(t, _bertscore_tok, cfg["MAX_TOKENS"]) for t in scoreable[cfg["REF_COL"]]]
    free_gpu()
    P, R, F1 = scorer.score(mt_texts, ref_texts)
    df.loc[scoreable.index, "bertscore_precision"] = P.numpy()
    df.loc[scoreable.index, "bertscore_recall"] = R.numpy()
    df.loc[scoreable.index, "bertscore_f1"] = F1.numpy()
    free_gpu()
    return df


# =========================================================================
# 4. LaBSE COSINE -- reuses a persistent model, internal truncation is safe
# =========================================================================
def compute_cosine_labse(df, cfg, labse):
    from sentence_transformers import util
    scoreable = df[df["has_pair"] == True]
    if len(scoreable) == 0:
        return df

    mt_texts = scoreable[cfg["MT_COL"]].tolist()
    ref_texts = scoreable[cfg["REF_COL"]].tolist()
    src_texts = scoreable[cfg["SOURCE_COL"]].tolist()

    emb_mt = labse.encode(mt_texts, batch_size=cfg["LABSE_BATCH_SIZE"], convert_to_tensor=True)
    emb_ref = labse.encode(ref_texts, batch_size=cfg["LABSE_BATCH_SIZE"], convert_to_tensor=True)
    emb_src = labse.encode(src_texts, batch_size=cfg["LABSE_BATCH_SIZE"], convert_to_tensor=True)

    df.loc[scoreable.index, "cosine_mt_vs_ref"] = util.pairwise_cos_sim(emb_mt, emb_ref).cpu().numpy()
    df.loc[scoreable.index, "cosine_src_vs_ref_direct"] = util.pairwise_cos_sim(emb_src, emb_ref).cpu().numpy()
    free_gpu()
    return df


# =========================================================================
# 5. COMET -- reuses a persistent model, hard token truncation
# =========================================================================
def compute_comet(df, cfg, comet_model):
    if not cfg["RUN_COMET"] or comet_model is None:
        return df
    scoreable = df[df["has_pair"] == True]
    if len(scoreable) == 0:
        return df

    data = [
        {
            "src": hard_truncate(s, _comet_tok, cfg["MAX_TOKENS"]),
            "mt": hard_truncate(m, _comet_tok, cfg["MAX_TOKENS"]),
            "ref": hard_truncate(r, _comet_tok, cfg["MAX_TOKENS"]),
        }
        for s, m, r in zip(scoreable[cfg["SOURCE_COL"]], scoreable[cfg["MT_COL"]], scoreable[cfg["REF_COL"]])
    ]
    free_gpu()
    out = comet_model.predict(data, batch_size=cfg["COMET_BATCH_SIZE"], gpus=1 if device == "cuda" else 0)
    df.loc[scoreable.index, "comet"] = out.scores
    free_gpu()
    return df


# =========================================================================
# 6. STRUCTURAL / ENTITY / NUMERIC CHECKS (unchanged, CPU regex only)
# =========================================================================
NUM_RE = re.compile(r"\d+(?:\.\d+)?")
DEVANAGARI_RE = re.compile(r"[\u0900-\u097F]")
CAP_ENTITY_RE = re.compile(r"\b[A-Z][A-Za-z]{2,}\b")

def extract_numbers(text):
    return set(NUM_RE.findall(text))

def numeric_consistency(src, mt):
    src_nums = extract_numbers(src)
    if not src_nums:
        return 1.0
    return len(src_nums & extract_numbers(mt)) / len(src_nums)

def extract_capitalized_entities(text):
    return set(w.upper() for w in CAP_ENTITY_RE.findall(text))

def entity_preservation(ref, mt):
    ref_ents = extract_capitalized_entities(ref)
    if not ref_ents:
        return 1.0
    return len(ref_ents & extract_capitalized_entities(mt)) / len(ref_ents)

def has_untranslated_source_script(mt_text):
    return bool(DEVANAGARI_RE.search(mt_text))

def length_ratio(src, mt):
    src_len = max(len(src.split()), 1)
    return len(mt.split()) / src_len

def compute_structural_checks(df, cfg):
    df["numeric_consistency"] = df.apply(
        lambda r: numeric_consistency(r[cfg["SOURCE_COL"]], r[cfg["MT_COL"]]), axis=1)
    scoreable = df[df["has_pair"] == True]
    df.loc[scoreable.index, "entity_preservation"] = scoreable.apply(
        lambda r: entity_preservation(r[cfg["REF_COL"]], r[cfg["MT_COL"]]), axis=1)
    df["has_untranslated_text"] = df[cfg["MT_COL"]].apply(has_untranslated_source_script)
    df["length_ratio"] = df.apply(
        lambda r: length_ratio(r[cfg["SOURCE_COL"]], r[cfg["MT_COL"]]), axis=1)
    return df


# =========================================================================
# 7. FLAGS (unchanged)
# =========================================================================
def compute_flags(df, cfg):
    if "bertscore_f1" in df:
        df["flag_low_bertscore"] = df["bertscore_f1"] < cfg["THRESH_BERTSCORE_F1"]
    if "cosine_mt_vs_ref" in df:
        df["flag_low_cosine"] = df["cosine_mt_vs_ref"] < cfg["THRESH_COSINE"]
    if "comet" in df:
        df["flag_low_comet"] = df["comet"] < cfg["THRESH_COMET"]
    if "chrf++" in df:
        df["flag_low_chrf"] = df["chrf++"] < cfg["THRESH_CHRF"]
    if "has_untranslated_text" in df:
        df["flag_untranslated_text"] = df["has_untranslated_text"]
    if "numeric_consistency" in df:
        df["flag_low_numeric_consistency"] = df["numeric_consistency"] < cfg["THRESH_NUMERIC_CONSISTENCY"]
    if "entity_preservation" in df:
        df["flag_low_entity_preservation"] = df["entity_preservation"] < cfg["THRESH_ENTITY_PRESERVATION"]
    return df


# =========================================================================
# 8. FINAL REPORT (unchanged)
# =========================================================================
def build_accuracy_report(df, cfg):
    scored = df[df["has_pair"] == True].copy()
    if len(scored) == 0:
        print("No scoreable rows -- skipping report.")
        return

    flagged_metrics = [
        ("chrf++", cfg["THRESH_CHRF"]),
        ("bertscore_f1", cfg["THRESH_BERTSCORE_F1"]),
        ("cosine_mt_vs_ref", cfg["THRESH_COSINE"]),
        ("numeric_consistency", cfg["THRESH_NUMERIC_CONSISTENCY"]),
        ("entity_preservation", cfg["THRESH_ENTITY_PRESERVATION"]),
    ]
    if cfg["RUN_COMET"]:
        flagged_metrics.insert(3, ("comet", cfg["THRESH_COMET"]))

    diagnostic_metrics = ["bleu", "ter", "rouge_l", "meteor", "cosine_src_vs_ref_direct", "length_ratio"]
    sections = sorted(scored["field_name"].unique())

    acc_rows = []
    for sec in sections:
        sub = scored[scored["field_name"] == sec]
        row = {"Section (Advisory)": sec, "n": len(sub)}
        for metric, thresh in flagged_metrics:
            if metric in sub.columns:
                row[metric] = round((sub[metric] >= thresh).mean() * 100, 1)
        acc_rows.append(row)
    acc_df = pd.DataFrame(acc_rows).set_index("Section (Advisory)")

    all_metric_cols = [m for m, _ in flagged_metrics] + diagnostic_metrics
    med_rows = []
    for sec in sections:
        sub = scored[scored["field_name"] == sec]
        row = {"Section (Advisory)": sec, "n": len(sub)}
        for metric in all_metric_cols:
            if metric in sub.columns:
                row[metric] = round(sub[metric].median(), 3)
        med_rows.append(row)
    med_df = pd.DataFrame(med_rows).set_index("Section (Advisory)")

    wb = Workbook()
    arial = "Arial"
    title_font = Font(name=arial, size=13, bold=True)
    header_font = Font(name=arial, size=10, bold=True)
    body_font = Font(name=arial, size=10)
    wrap = Alignment(wrap_text=True, vertical="top", horizontal="left")
    center = Alignment(wrap_text=True, vertical="top", horizontal="center")
    thin = Side(style="thin", color="000000")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)

    def write_table(ws, title, table_df, subtitle=""):
        ws.merge_cells(f"A1:{get_column_letter(len(table_df.columns)+1)}1")
        ws["A1"] = title
        ws["A1"].font = title_font
        ws.row_dimensions[1].height = 22
        r0 = 2
        if subtitle:
            ws.merge_cells(f"A2:{get_column_letter(len(table_df.columns)+1)}2")
            ws["A2"] = subtitle
            ws["A2"].font = Font(name=arial, size=9, italic=True)
            r0 = 3
        headers = ["Section (Advisory)"] + list(table_df.columns)
        for c, h in enumerate(headers, start=1):
            cell = ws.cell(row=r0, column=c, value=h)
            cell.font = header_font
            cell.border = border
            cell.alignment = center
        r = r0 + 1
        for idx, row in table_df.iterrows():
            ws.cell(row=r, column=1, value=idx).font = body_font
            ws.cell(row=r, column=1).border = border
            ws.cell(row=r, column=1).alignment = wrap
            for c, val in enumerate(row, start=2):
                cell = ws.cell(row=r, column=c, value=val)
                cell.font = body_font
                cell.border = border
                cell.alignment = center
            r += 1
        widths = [26] + [16] * len(table_df.columns)
        for i, w in enumerate(widths, start=1):
            ws.column_dimensions[get_column_letter(i)].width = w
        ws.freeze_panes = f"A{r0+1}"

    ws1 = wb.active
    ws1.title = "Accuracy % by Section"
    write_table(ws1, f"Accuracy % by Advisory Section ({SOURCE_LANG_NAME} -> English)",
                acc_df, "Each cell = % of that section's rows meeting the metric's accuracy threshold.")

    ws2 = wb.create_sheet("Median Scores by Section")
    write_table(ws2, "Median Raw Score by Advisory Section (per metric)",
                med_df, "Diagnostic metrics shown as median only -- not used for pass/fail.")

    ws3 = wb.create_sheet("Metric Definitions")
    ws3["A1"] = "Metric Definitions and Thresholds"
    ws3["A1"].font = title_font
    defs = [
        ("bleu", "0-100", "Diagnostic only", "Word n-gram overlap. Naturally low for paraphrastic MT."),
        ("chrf++", "0-100", f">= {cfg['THRESH_CHRF']}", "Character n-gram overlap; more forgiving than BLEU."),
        ("ter", "0-100+ (lower=better)", "Diagnostic only", "Edit distance to fix MT into reference."),
        ("rouge_l", "0-1", "Diagnostic only", "Longest common word-sequence overlap."),
        ("meteor", "0-1", "Diagnostic only", "Overlap with stemming/synonym matching."),
        ("bertscore_f1", "0-1", f">= {cfg['THRESH_BERTSCORE_F1']}", "Embedding-based semantic overlap."),
        ("cosine_mt_vs_ref", "0-1 (theoretical -1 to 1)", f">= {cfg['THRESH_COSINE']}", "LaBSE meaning-similarity, MT vs gold reference."),
        ("cosine_src_vs_ref_direct", "0-1 (theoretical -1 to 1)", "Diagnostic only (checks reference, not model)", f"LaBSE similarity, {SOURCE_LANG_NAME} source vs gold reference directly."),
        ("comet", "Roughly 0-1", f">= {cfg['THRESH_COMET']}" if cfg["RUN_COMET"] else "Skipped this run", "Trained quality-estimation model; closest to human judgment."),
        ("numeric_consistency", "0-1", f">= {cfg['THRESH_NUMERIC_CONSISTENCY']}", "Fraction of source numbers preserved in MT."),
        ("entity_preservation", "0-1", f">= {cfg['THRESH_ENTITY_PRESERVATION']}", "Fraction of reference's named entities found in MT."),
        ("length_ratio", "Roughly 0-2 (1.0=same length)", "Diagnostic only", "MT word count / source word count."),
    ]
    headers = ["Metric", "Range", "Accuracy Threshold", "What It Measures"]
    for c, h in enumerate(headers, start=1):
        cell = ws3.cell(row=3, column=c, value=h)
        cell.font = header_font
        cell.border = border
        cell.alignment = center
    r = 4
    for metric, scale, thresh, desc in defs:
        vals = [metric, scale, thresh, desc]
        for c, v in enumerate(vals, start=1):
            cell = ws3.cell(row=r, column=c, value=v)
            cell.font = body_font
            cell.border = border
            cell.alignment = wrap
        ws3.row_dimensions[r].height = 32
        r += 1
    widths = [22, 26, 30, 55]
    for i, w in enumerate(widths, start=1):
        ws3.column_dimensions[get_column_letter(i)].width = w
    ws3.freeze_panes = "A4"

    wb.save(cfg["REPORT_OUTPUT_FILE"])
    print(f"\nSaved {cfg['REPORT_OUTPUT_FILE']}")
    print("\n--- Accuracy % by Section ---")
    print(acc_df)


# =========================================================================
# MAIN -- section-by-section, checkpointed, models loaded ONCE
# =========================================================================
def run_validation_pipeline(cfg):
    overall_start = time.time()

    print("=== STEP 1: Merging gold references ===")
    df = merge_gold_references(cfg)
    sections = list(df["field_name"].unique())

    print("\n=== Loading models once (reused across all sections) ===")
    from bert_score import BERTScorer
    from sentence_transformers import SentenceTransformer

    bertscorer = BERTScorer(lang=TARGET_LANG_CODE, model_type=cfg["BERTSCORE_MODEL"],
                             device=device, batch_size=cfg["BERTSCORE_BATCH_SIZE"])
    labse = SentenceTransformer("sentence-transformers/LaBSE", device=device)

    comet_model = None
    if cfg["RUN_COMET"]:
        from comet import download_model, load_from_checkpoint
        print("Loading COMET (reference-based)...")
        path = download_model("Unbabel/wmt22-comet-da")
        comet_model = load_from_checkpoint(path)

    all_results = []
    for i, sec in enumerate(sections, 1):
        ckpt_path = os.path.join(CHECKPOINT_DIR, f"{sec}.xlsx")
        if os.path.exists(ckpt_path):
            print(f"\n=== Section {i}/{len(sections)}: {sec} -- already checkpointed, loading from disk ===")
            sub = pd.read_excel(ckpt_path)
            all_results.append(sub)
            continue

        print(f"\n=== Section {i}/{len(sections)}: {sec} ===")
        sub = df[df["field_name"] == sec].copy()
        n_scoreable = sub["has_pair"].sum()
        print(f"  {n_scoreable} scoreable rows")

        t0 = time.time()
        sub = compute_ngram_metrics(sub, cfg)
        print(f"  n-gram metrics done ({time.time()-t0:.1f}s)")

        t0 = time.time()
        sub = compute_bertscore(sub, cfg, bertscorer)
        print(f"  BERTScore done ({time.time()-t0:.1f}s)")

        t0 = time.time()
        sub = compute_cosine_labse(sub, cfg, labse)
        print(f"  LaBSE cosine done ({time.time()-t0:.1f}s)")

        t0 = time.time()
        sub = compute_comet(sub, cfg, comet_model)
        print(f"  COMET done ({time.time()-t0:.1f}s)")

        sub = compute_structural_checks(sub, cfg)
        sub = compute_flags(sub, cfg)

        sub.to_excel(ckpt_path, index=False)
        print(f"  Checkpoint saved: {ckpt_path}")

        all_results.append(sub)
        del sub
        free_gpu()

    del bertscorer, labse, comet_model
    free_gpu()

    final_df = pd.concat(all_results, ignore_index=True)
    final_df.to_excel(cfg["RESULTS_OUTPUT_FILE"], index=False)
    print(f"\nSaved {cfg['RESULTS_OUTPUT_FILE']}")

    print("\n=== Building final accuracy report ===")
    build_accuracy_report(final_df, cfg)

    print(f"\nAll done in {(time.time()-overall_start)/60:.1f} min total.")
    return final_df


# ---- RUN ----
final_df = run_validation_pipeline(CONFIG)

Using device: cuda
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Outputs -> /content/drive/MyDrive/agromet_pipeline_outputs
Checkpoints -> /content/drive/MyDrive/agromet_pipeline_outputs/section_checkpoints
Loading tokenizers for truncation...
=== STEP 1: Merging gold references ===
Sections found: ['forecast_summary', 'bulletin_subtitle', 'weather_warnings', 'weather_warning_impacts', 'weather_impacts_general', 'general_advisory', 'sms_advisory', 'impact_based_advisories', 'crop_advisory', 'horticulture_advisory', 'livestock_advisory', 'fisheries_advisory', 'poultry_advisory', 'masthead', 'preamble', 'unmatched_other']

  forecast_summary                 rows=  559  paired=  520 (93.0%)
  bulletin_subtitle                rows=  560  paired=  523 (93.4%)
  weather_warnings                 rows=  180  paired=  168 (93.3%)
  weather_warning_impacts          rows=  180  paired=  168 (93.3%)
  weather_impact

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../root/.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']



=== Section 1/16: forecast_summary -- already checkpointed, loading from disk ===

=== Section 2/16: bulletin_subtitle -- already checkpointed, loading from disk ===

=== Section 3/16: weather_warnings -- already checkpointed, loading from disk ===

=== Section 4/16: weather_warning_impacts -- already checkpointed, loading from disk ===

=== Section 5/16: weather_impacts_general -- already checkpointed, loading from disk ===

=== Section 6/16: general_advisory -- already checkpointed, loading from disk ===

=== Section 7/16: sms_advisory -- already checkpointed, loading from disk ===

=== Section 8/16: impact_based_advisories -- already checkpointed, loading from disk ===

=== Section 9/16: crop_advisory -- already checkpointed, loading from disk ===

=== Section 10/16: horticulture_advisory -- already checkpointed, loading from disk ===

=== Section 11/16: livestock_advisory -- already checkpointed, loading from disk ===

=== Section 12/16: fisheries_advisory -- already checkpointed,

In [3]:
import pandas as pd

def generate_simple_accuracy_report(final_df, cfg,
                                    output_file="Simple_Section_Accuracy_Report.xlsx"):

    scored = final_df[final_df["has_pair"] == True].copy()

    metric_info = [
        ("chrf++", "≥40", cfg["THRESH_CHRF"]),
        ("bertscore_f1", "≥0.80", cfg["THRESH_BERTSCORE_F1"]),
        ("cosine_mt_vs_ref", "≥0.75", cfg["THRESH_COSINE"]),
        ("comet", "≥0.75", cfg["THRESH_COMET"]),
        ("numeric_consistency", "≥0.50", cfg["THRESH_NUMERIC_CONSISTENCY"]),
        ("entity_preservation", "≥0.40", cfg["THRESH_ENTITY_PRESERVATION"]),
    ]

    rows = []

    for section in sorted(scored["field_name"].unique()):

        sub = scored[scored["field_name"] == section]

        total_rows = len(sub)

        overall = []

        for metric, threshold_text, threshold_value in metric_info:

            if metric not in sub.columns:
                continue

            accuracy = round((sub[metric] >= threshold_value).mean() * 100, 1)

            overall.append(accuracy)

            if accuracy >= 90:
                interpretation = "Excellent"
                remarks = "Very high quality. Most rows satisfy the validation threshold."

            elif accuracy >= 80:
                interpretation = "Very Good"
                remarks = "Translation quality is reliable with minor deviations."

            elif accuracy >= 70:
                interpretation = "Good"
                remarks = "Generally acceptable but a few rows require review."

            elif accuracy >= 50:
                interpretation = "Fair"
                remarks = "Moderate performance. Manual verification is recommended."

            else:
                interpretation = "Needs Improvement"
                remarks = "Large number of rows failed the validation threshold."

            rows.append({
                "Section": section,
                "Rows Evaluated": total_rows,
                "Metric": metric,
                "Threshold": threshold_text,
                "Accuracy (%)": accuracy,
                "Interpretation": interpretation,
                "Remarks": remarks
            })

        overall_accuracy = round(sum(overall)/len(overall),1)

        if overall_accuracy >= 90:
            overall_interpretation = "Excellent"

        elif overall_accuracy >= 80:
            overall_interpretation = "Very Good"

        elif overall_accuracy >= 70:
            overall_interpretation = "Good"

        elif overall_accuracy >= 50:
            overall_interpretation = "Fair"

        else:
            overall_interpretation = "Needs Improvement"

        rows.append({
            "Section": section,
            "Rows Evaluated": total_rows,
            "Metric": "Overall Section Accuracy",
            "Threshold": "-",
            "Accuracy (%)": overall_accuracy,
            "Interpretation": overall_interpretation,
            "Remarks": "Average accuracy across all evaluation metrics."
        })

    report = pd.DataFrame(rows)

    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

        report.to_excel(writer,
                        sheet_name="Accuracy Report",
                        index=False)

        workbook = writer.book
        worksheet = writer.sheets["Accuracy Report"]

        from openpyxl.styles import Font, PatternFill, Alignment

        header_fill = PatternFill(fill_type="solid",
                                  fgColor="1F4E78")

        header_font = Font(color="FFFFFF",
                           bold=True)

        for cell in worksheet[1]:
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal="center")

        widths = {
            "A":28,
            "B":16,
            "C":30,
            "D":12,
            "E":14,
            "F":20,
            "G":55
        }

        for col,width in widths.items():
            worksheet.column_dimensions[col].width = width

    print(f"\nReport Saved Successfully -> {output_file}")


In [4]:
generate_simple_accuracy_report(
    final_df,
    CONFIG,
    output_file="Section_Metric_Accuracy_Report.xlsx"
)


Report Saved Successfully -> Section_Metric_Accuracy_Report.xlsx
